In [ ]:
!pip install -q supervision transformers timm tqdm

In [ ]:
#CONNECT GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
HOME = "/content/drive/MyDrive/Updated DETR Model"
MODEL_PATH = os.path.join(HOME, 'custom-modelv2-50epochs')

In [ ]:
### vanilla particle filter
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from transformers import DetrForObjectDetection, DetrImageProcessor
from scipy.optimize import linear_sum_assignment
import supervision as sv
import torchvision

video_path = '/content/drive/MyDrive/data record/DRON/01/DJI_0006.MP4'
HOME = "/content/drive/MyDrive/Updated DETR Model"
model_dir = os.path.join(HOME, 'custom-modelv2-50epochs')
output_directory = '/content/drive/MyDrive/ParticleFilter'
output_filename = "output.mp4"
output_video_path = os.path.join(output_directory, output_filename)
os.makedirs(output_directory, exist_ok=True)

FRAME_SKIP = 1
MAX_SECONDS_TO_PROCESS = 10
GRID_SIZE = (4, 4)
TILE_TO_TRACK = (0, 0)
CONFIDENCE_THRESHOLD = 0.8
NMS_IOU_THRESHOLD = 0.5
TRACKING_IOU_THRESHOLD = 0.6
MAX_INVISIBLE_COUNT = 30
NUM_PARTICLES = 250
PROCESS_NOISE_STD = {'x': 1, 'y': 1, 'vx': 1, 'vy': 1, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained(model_dir, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(model_dir)
model.eval()



class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std):
        self.num_particles, self.measurement_noise_std = num_particles, measurement_noise_std
        self.particles = np.empty((self.num_particles, self.STATE_DIM))
        self.weights = np.ones(self.num_particles) / self.num_particles
        self.proc_noise = np.array([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']])

    def initialize(self, bbox):
        cx, cy, w, h = bbox[0] + bbox[2]/2, bbox[1] + bbox[3]/2, bbox[2], bbox[3]
        initial_state = np.array([cx, cy, 0, 0, w, h])
        self.particles = initial_state + np.random.randn(self.num_particles, self.STATE_DIM) * self.proc_noise

    def predict(self):
        A = np.array([[1, 0, 1, 0, 0, 0], [0, 1, 0, 1, 0, 0], [0, 0, 1, 0, 0, 0],
                      [0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1]])
        self.particles = self.particles @ A.T + np.random.randn(self.num_particles, self.STATE_DIM) * self.proc_noise

    def update(self, bbox):
        det_cx, det_cy = bbox[0] + bbox[2]/2, bbox[1] + bbox[3]/2
        distances = np.sqrt((self.particles[:, 0] - det_cx)**2 + (self.particles[:, 1] - det_cy)**2)
        likelihood = np.exp(-0.5 * (distances / self.measurement_noise_std)**2)
        self.weights = (self.weights * likelihood) + 1e-300
        self.weights /= np.sum(self.weights)

    def resample(self):
        indices = np.random.choice(np.arange(self.num_particles), size=self.num_particles, p=self.weights)
        self.particles = self.particles[indices]
        self.weights.fill(1.0 / self.num_particles)

    def get_state(self):
        return np.average(self.particles, weights=self.weights, axis=0)

def get_color_histogram(frame, bbox):
    x, y, w, h = [int(v) for v in bbox]
    roi = frame[max(0,y):y+h, max(0,x):x+w]
    if roi.size == 0: return None
    hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv_roi], [0, 1], None, [16, 16], [0, 180, 0, 256])
    return cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)

class Tracker:
    def __init__(self, track_id, initial_bbox, initial_frame):
        self.id = track_id
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD)
        self.pf.initialize(initial_bbox)
        self.invisible_count = 0
        self.histogram = get_color_histogram(initial_frame, initial_bbox)

def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1; x2, y2, w2, h2 = box2
    xi1, yi1, xi2, yi2 = max(x1, x2), max(y1, y2), min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = (w1 * h1) + (w2 * h2) - inter
    return inter / (union + 1e-6)

def run_detector(frame_bgr, model, image_processor, device):
    image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = image_processor(images=image_rgb, return_tensors='pt').to(device)
        outputs = model(**inputs)
        res = image_processor.post_process_object_detection(outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=torch.tensor([frame_bgr.shape[:2]]).to(device))[0]
    if not res['scores'].numel(): return []
    raw = sv.Detections.from_transformers(transformers_results=res)
    keep = torchvision.ops.nms(torch.from_numpy(raw.xyxy).to(device), torch.from_numpy(raw.confidence).to(device), NMS_IOU_THRESHOLD).cpu().tolist()
    xyxy = raw.xyxy[keep]
    return [np.array([b[0], b[1], b[2]-b[0], b[3]-b[1]]) for b in xyxy]

cap = cv2.VideoCapture(video_path)
W, H, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
total_frames = min(int(cap.get(7)), int(MAX_SECONDS_TO_PROCESS * fps)) if MAX_SECONDS_TO_PROCESS else int(cap.get(7))
th, tw = H // GRID_SIZE[0], W // GRID_SIZE[1]
oy, ox = TILE_TO_TRACK[0] * th, TILE_TO_TRACK[1] * tw
out_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/FRAME_SKIP, (tw, th))

pf_trackers, next_tid = [], 0

for fidx in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret: break
    if fidx % FRAME_SKIP == 0:
        crop = frame[oy:oy+th, ox:ox+tw].copy()
        dets = run_detector(crop, model, processor, device)
        for t in pf_trackers: t.pf.predict()
        if pf_trackers and dets:
            costs = np.zeros((len(pf_trackers), len(dets)))
            for i, t in enumerate(pf_trackers):
                s = t.pf.get_state()
                t_box = [s[0]-s[4]/2, s[1]-s[5]/2, s[4], s[5]]
                for j, d_box in enumerate(dets):
                    spatial = 1 - calculate_iou(t_box, d_box)
                    d_hist = get_color_histogram(crop, d_box)
                    app = cv2.compareHist(t.histogram, d_hist, 3) if t.histogram is not None and d_hist is not None else 1.0
                    costs[i, j] = 0.7 * spatial + 0.3 * app
            r_idx, c_idx = linear_sum_assignment(costs)
            matched, avail_d = set(), set(range(len(dets)))
            for r, c in zip(r_idx, c_idx):
                if costs[r, c] < (1 - TRACKING_IOU_THRESHOLD):
                    pf_trackers[r].pf.update(dets[c]); pf_trackers[r].invisible_count = 0
                    matched.add(r); avail_d.discard(c)
            for i in set(range(len(pf_trackers))) - matched: pf_trackers[i].invisible_count += 1
            for i in avail_d: pf_trackers.append(Tracker(next_tid, dets[i], crop)); next_tid += 1
        elif dets:
            for d in dets: pf_trackers.append(Tracker(next_tid, d, crop)); next_tid += 1
        pf_trackers = [t for t in pf_trackers if t.invisible_count <= MAX_INVISIBLE_COUNT]
        for t in pf_trackers:
            if t.invisible_count == 0:
                t.pf.resample()
                s = t.pf.get_state()
                p1, p2 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2)), (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
                cv2.rectangle(crop, p1, p2, (0, 255, 0), 2)
                cv2.putText(crop, f"ID {t.id}", (p1[0], p1[1]-10), 0, 0.7, (0, 255, 0), 2)
        out_video.write(crop)

cap.release()
out_video.release()

In [ ]:
### GPU-Optimized Repulsive Force Particle Filter
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from transformers import DetrForObjectDetection, DetrImageProcessor
from scipy.optimize import linear_sum_assignment
import supervision as sv
import torchvision

video_path = '/content/drive/MyDrive/data record/DRON/01/DJI_0006.MP4'
HOME = "/content/drive/MyDrive/Updated DETR Model"
model_dir = os.path.join(HOME, 'custom-modelv2-50epochs')
output_directory = '/content/drive/MyDrive/ParticleFilter'
output_filename = "output_APF_GPU_Optimized.mp4"
output_video_path = os.path.join(output_directory, output_filename)
os.makedirs(output_directory, exist_ok=True)

FRAME_SKIP = 1
MAX_SECONDS_TO_PROCESS = 10
GRID_SIZE = (4, 4)
TILE_TO_TRACK = (0, 0)
CONFIDENCE_THRESHOLD = 0.8
NMS_IOU_THRESHOLD = 0.5
TRACKING_IOU_THRESHOLD = 0.6
MAX_INVISIBLE_COUNT = 30
NUM_PARTICLES = 250
PROCESS_NOISE_STD = {'x': 1, 'y': 1, 'vx': 1, 'vy': 1, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 20
SIGMA_REPULSION = 50.0

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device = device
        self.num_particles = num_particles
        self.measurement_noise_std = measurement_noise_std
        self.particles = torch.empty((self.num_particles, self.STATE_DIM), device=self.device)
        self.weights = torch.ones(self.num_particles, device=self.device) / self.num_particles
        self.proc_noise = torch.tensor([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=self.device).float()

    def initialize(self, bbox_tensor):
        cx, cy, w, h = bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2, bbox_tensor[2], bbox_tensor[3]
        initial_state = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float()
        self.particles = initial_state + torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self, other_vehicle_states_tensor):
        F_att = self.particles[:, 2:4]
        F_rb_total = torch.zeros((self.num_particles, 2), device=self.device)
        if other_vehicle_states_tensor.numel() > 0:
            particle_pos = self.particles[:, 0:2]
            other_pos = other_vehicle_states_tensor[:, 0:2]
            vec_to_other = other_pos.unsqueeze(0) - particle_pos.unsqueeze(1)
            distances = torch.clamp(torch.norm(vec_to_other, dim=2), min=1e-6)
            force_magnitude = torch.exp(-(distances**2) / (SIGMA_REPULSION**2))
            F_rb_total = torch.sum(force_magnitude.unsqueeze(2) * (vec_to_other / distances.unsqueeze(2)), dim=1)
        resultant_force = F_att - F_rb_total
        self.particles[:, 0:2] += resultant_force
        self.particles += torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def update(self, bbox_tensor):
        det_c = torch.tensor([bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2], device=self.device)
        distances = torch.norm(self.particles[:, 0:2] - det_c, dim=1)
        self.weights = (self.weights * torch.exp(-0.5 * (distances / self.measurement_noise_std)**2)) + 1e-300
        self.weights /= torch.sum(self.weights)

    def resample(self):
        indices = torch.multinomial(self.weights, self.num_particles, replacement=True)
        self.particles = self.particles[indices]
        self.weights.fill_(1.0 / self.num_particles)

    def get_state(self):
        return torch.sum(self.particles * self.weights.unsqueeze(1), dim=0)

class Tracker:
    def __init__(self, track_id, initial_bbox_tensor, device):
        self.id = track_id
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, device)
        self.pf.initialize(initial_bbox_tensor)
        self.invisible_count = 0

def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1; x2, y2, w2, h2 = box2
    xi1, yi1 = max(x1, x2), max(y1, y2); xi2, yi2 = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = (w1 * h1) + (w2 * h2) - inter
    return inter / (union + 1e-6)

def run_detector(frame_bgr, model, image_processor, device):
    image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = image_processor(images=image_rgb, return_tensors='pt').to(device)
        outputs = model(**inputs)
        res = image_processor.post_process_object_detection(outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=torch.tensor([frame_bgr.shape[:2]], device=device))[0]
    if len(res['scores']) == 0: return torch.empty((0, 4), device=device)
    boxes, scores = res['boxes'], res['scores']
    keep = torchvision.ops.nms(boxes, scores, NMS_IOU_THRESHOLD)
    b = boxes[keep]
    return torch.stack([b[:, 0], b[:, 1], b[:, 2]-b[:, 0], b[:, 3]-b[:, 1]], dim=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained(model_dir, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(model_dir)
model.eval()

cap = cv2.VideoCapture(video_path)
W, H, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
total_f = min(int(cap.get(7)), int(MAX_SECONDS_TO_PROCESS * fps)) if MAX_SECONDS_TO_PROCESS else int(cap.get(7))
tile_h, tile_w = H // GRID_SIZE[0], W // GRID_SIZE[1]
oy, ox = TILE_TO_TRACK[0] * tile_h, TILE_TO_TRACK[1] * tile_w
out_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/FRAME_SKIP, (tile_w, tile_h))

pf_trackers, next_tid = [], 0

for fidx in tqdm(range(total_f)):
    ret, frame = cap.read()
    if not ret: break
    if fidx % FRAME_SKIP == 0:
        crop = frame[oy:oy+tile_h, ox:ox+tile_w].copy()
        dets = run_detector(crop, model, processor, device)
        if pf_trackers:
            states = torch.stack([t.pf.get_state() for t in pf_trackers])
            for i, t in enumerate(pf_trackers):
                others = states[[j for j in range(len(pf_trackers)) if i != j]] if len(pf_trackers) > 1 else torch.empty((0, 6), device=device)
                t.pf.predict(others)
        if pf_trackers and dets.numel() > 0:
            st = torch.stack([t.pf.get_state() for t in pf_trackers]).cpu().numpy()
            t_box = np.stack([st[:,0]-st[:,4]/2, st[:,1]-st[:,5]/2, st[:,4], st[:,5]], axis=1)
            d_box = dets.cpu().numpy()
            cost = 1 - np.array([[calculate_iou(t, d) for d in d_box] for t in t_box])
            r_idx, c_idx = linear_sum_assignment(cost)
            matched, avail_d = set(), set(range(len(d_box)))
            for r, c in zip(r_idx, c_idx):
                if cost[r, c] < (1 - TRACKING_IOU_THRESHOLD):
                    pf_trackers[r].pf.update(dets[c]); pf_trackers[r].invisible_count = 0
                    matched.add(r); avail_d.discard(c)
            for i in set(range(len(pf_trackers))) - matched: pf_trackers[i].invisible_count += 1
            for i in avail_d: pf_trackers.append(Tracker(next_tid, dets[i], device)); next_tid += 1
        elif dets.numel() > 0:
            for i in range(dets.shape[0]): pf_trackers.append(Tracker(next_tid, dets[i], device)); next_tid += 1
        pf_trackers = [t for t in pf_trackers if t.invisible_count <= MAX_INVISIBLE_COUNT]
        for t in pf_trackers:
            if t.invisible_count == 0:
                t.pf.resample()
                s = t.pf.get_state().cpu().numpy()
                p1, p2 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2)), (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
                cv2.rectangle(crop, p1, p2, (0, 255, 0), 2)
                cv2.putText(crop, f"ID {t.id}", (p1[0], p1[1]-10), 0, 0.7, (0, 255, 0), 2)
        out_video.write(crop)
cap.release()
out_video.release()

In [ ]:
###Applying velocity instead of accleration
### Repulsive Force Particle Filter (Boundary Heuristic)
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from transformers import DetrForObjectDetection, DetrImageProcessor
import supervision as sv
import torchvision
import math
import glob
import json

video_path = '/content/drive/MyDrive/croppedVideoForAnnotation.MP4'
HOME = "/content/drive/MyDrive/Updated DETR Model"
model_dir = os.path.join(HOME, 'custom-modelv2-50epochs')
output_directory = '/content/drive/MyDrive/ParticleFilter'
output_filename = "output_APF_GPU_Divider_Boundary_Heuristic.mp4"
output_video_path = os.path.join(output_directory, output_filename)
os.makedirs(output_directory, exist_ok=True)
COCO_ANNOTATIONS_FILE_PATH = '/content/drive/MyDrive/ParticleFilterWithPartition/train/_annotations.coco.json'

FRAME_ID_TO_LOAD = 0
FRAME_SKIP = 1
MAX_SECONDS_TO_PROCESS = 10
DIVIDER_P1 = (0, 104)
DIVIDER_P2 = (496.809, 377.316)
ROAD_ANGLE_NORTH = 28.8
ROAD_ANGLE_SOUTH = 208.8
CONFIDENCE_THRESHOLD = 0.95
NMS_IOU_THRESHOLD = 0.5
IOU_MATCH_THRESHOLD = 0.1
MAX_INVISIBLE_COUNT = 6
BOUNDARY_PERCENT = 0.05
NUM_PARTICLES = 75
PROCESS_NOISE_STD = {'x': 2, 'y': 2, 'vx': 1.5, 'vy': 2, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 20
SIGMA_REPULSION = 40.0

def is_in_boundary(center_point_tensor, W, H, percent):
    bound_x, bound_y = W * percent, H * percent
    cx, cy = center_point_tensor[0], center_point_tensor[1]
    return (cx < bound_x) or (cx > W - bound_x) or (cy < bound_y) or (cy > H - bound_y)

def is_north_of_line(point_tensor, line_p1, line_p2):
    x, y = point_tensor[:, 0], point_tensor[:, 1]
    x1, y1 = line_p1; x2, y2 = line_p2
    return ((x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)) > 0

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device, self.num_particles, self.measurement_noise_std = device, num_particles, measurement_noise_std
        self.particles = torch.empty((num_particles, self.STATE_DIM), device=device)
        self.weights = torch.ones(num_particles, device=device) / num_particles
        self.proc_noise = torch.tensor([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=device).float()

    def initialize(self, bbox_tensor):
        cx, cy, w, h = bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2, bbox_tensor[2], bbox_tensor[3]
        initial = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float()
        self.particles = initial + torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self, other_states, ang_n, ang_s, p1, p2):
        pos = self.particles[:, 0:2]
        angs = torch.where(is_north_of_line(pos, p1, p2), ang_n, ang_s)
        V_des = torch.stack([2.0 * torch.cos(angs), 2.0 * torch.sin(angs)], dim=1)
        V_rep = torch.zeros((self.num_particles, 2), device=self.device)
        if other_states.numel() > 0:
            vecs = other_states[:, 0:2].unsqueeze(0) - pos.unsqueeze(1)
            dist = torch.clamp(torch.norm(vecs, dim=2), min=1e-6)
            force = torch.exp(-(dist**2) / (SIGMA_REPULSION**2))
            V_rep = -torch.sum(force.unsqueeze(2) * (vecs / dist.unsqueeze(2)), dim=1)
        target_v = V_des + V_rep
        alpha = 0.5
        self.particles[:, 2:4] = (1 - alpha) * self.particles[:, 2:4] + alpha * torch.nan_to_num(target_v)
        self.particles[:, 0:2] += self.particles[:, 2:4]
        self.particles += torch.randn_like(self.particles) * self.proc_noise

    def update(self, bbox):
        det_c = torch.tensor([bbox[0] + bbox[2]/2, bbox[1] + bbox[3]/2], device=self.device)
        likelihood = torch.exp(-0.5 * (torch.norm(self.particles[:, 0:2] - det_c, dim=1) / self.measurement_noise_std)**2)
        self.weights = (self.weights * torch.nan_to_num(likelihood, nan=1e-10)) + 1e-300
        self.weights /= torch.sum(self.weights)

    def resample(self):
        self.weights = torch.nan_to_num(torch.clamp(self.weights, min=0.0), nan=1.0/self.num_particles)
        s = torch.sum(self.weights)
        w = self.weights / s if s > 0 else torch.ones_like(self.weights)/self.num_particles
        idx = torch.multinomial(w, self.num_particles, replacement=True)
        self.particles, self.weights = self.particles[idx], torch.ones(self.num_particles, device=self.device)/self.num_particles

    def get_state(self):
        w = torch.nan_to_num(self.weights, nan=0.0)
        p = torch.nan_to_num(self.particles, nan=0.0)
        s = torch.sum(w)
        return torch.sum(p * (w/s).unsqueeze(1), dim=0) if s > 0 else torch.mean(p, dim=0)

class Tracker:
    def __init__(self, tid, bbox, dev):
        self.id, self.invisible_count = tid, 0
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, dev)
        self.pf.initialize(bbox)

def calculate_iou_torch(b1, b2):
    x1y1_1, x2y2_1 = b1[:, :2], b1[:, :2] + b1[:, 2:]
    x1y1_2, x2y2_2 = b2[:, :2], b2[:, :2] + b2[:, 2:]
    xi1, yi1 = torch.max(x1y1_1[:, 0].unsqueeze(1), x1y1_2[:, 0].unsqueeze(0)), torch.max(x1y1_1[:, 1].unsqueeze(1), x1y1_2[:, 1].unsqueeze(0))
    xi2, yi2 = torch.min(x2y2_1[:, 0].unsqueeze(1), x2y2_2[:, 0].unsqueeze(0)), torch.min(x2y2_1[:, 1].unsqueeze(1), x2y2_2[:, 1].unsqueeze(0))
    inter = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    union = (b1[:, 2] * b1[:, 3]).unsqueeze(1) + (b2[:, 2] * b2[:, 3]).unsqueeze(0) - inter
    return torch.nan_to_num(inter / (union + 1e-6), nan=0.0)

def run_detector(frame, model, proc, dev):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = proc(images=rgb, return_tensors='pt').to(dev)
        outputs = model(**inputs)
        res = proc.post_process_object_detection(outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=torch.tensor([frame.shape[:2]], device=dev))[0]
    if not res['scores'].numel(): return torch.empty((0, 4), device=dev)
    raw = sv.Detections.from_transformers(transformers_results=res)
    boxes, scores = torch.from_numpy(raw.xyxy).to(dev), torch.from_numpy(raw.confidence).to(dev)
    keep = torchvision.ops.nms(boxes, scores, iou_threshold=NMS_IOU_THRESHOLD)
    xy = raw.xyxy[keep.cpu().numpy()]
    return torch.from_numpy(np.stack([xy[:, 0], xy[:, 1], xy[:, 2]-xy[:, 0], xy[:, 3]-xy[:, 1]], axis=1)).float().to(dev)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained(model_dir, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(model_dir)
model.eval()

cap = cv2.VideoCapture(video_path)
W, H, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
total_f = min(int(cap.get(7)), int(MAX_SECONDS_TO_PROCESS * fps))
out_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/FRAME_SKIP, (W, H))

ang_n, ang_s = torch.deg2rad(torch.tensor(ROAD_ANGLE_NORTH, device=device)), torch.deg2rad(torch.tensor(ROAD_ANGLE_SOUTH, device=device))
dp1, dp2 = torch.tensor(DIVIDER_P1, device=device).float(), torch.tensor(DIVIDER_P2, device=device).float()
pf_trackers, next_tid = [], 0

for fidx in tqdm(range(total_f)):
    ret, frame = cap.read()
    if not ret: break
    if fidx % FRAME_SKIP == 0:
        current_frame = frame.copy()
        if fidx == 0:
            with open(COCO_ANNOTATIONS_FILE_PATH, 'r') as f:
                anns = [a['bbox'] for a in json.load(f).get('annotations', []) if a.get('image_id') == FRAME_ID_TO_LOAD]
            dets = torch.tensor(anns, dtype=torch.float32).to(device)
            for i in range(dets.shape[0]):
                pf_trackers.append(Tracker(next_tid, dets[i], device)); next_tid += 1
        else:
            dets = run_detector(current_frame, model, processor, device)
            if pf_trackers:
                st_all = torch.stack([t.pf.get_state() for t in pf_trackers])
                for i, t in enumerate(pf_trackers):
                    others = st_all[[j for j in range(len(pf_trackers)) if i != j]] if len(pf_trackers) > 1 else torch.empty((0, 6), device=device)
                    t.pf.predict(others, ang_n, ang_s, dp1, dp2)

            to_del = set()
            if pf_trackers and dets.numel() > 0:
                st = torch.stack([t.pf.get_state() for t in pf_trackers])
                t_b = torch.stack([st[:,0]-st[:,4]/2, st[:,1]-st[:,5]/2, st[:,4], st[:,5]], axis=1)
                iou_c = (1.0 - calculate_iou_torch(t_b, dets)).cpu().numpy()
                t_idx, d_idx = np.unravel_index(np.argsort(iou_c, axis=None), iou_c.shape)
                matched, avail_d, used_t = set(), set(range(dets.shape[0])), set()
                for r, c in zip(t_idx, d_idx):
                    if r not in used_t and c in avail_d and (1.0 - iou_c[r, c]) > IOU_MATCH_THRESHOLD:
                        pf_trackers[r].pf.update(dets[c]); pf_trackers[r].invisible_count = 0
                        matched.add(r); used_t.add(r); avail_d.remove(c)
                for i in (set(range(len(pf_trackers))) - matched):
                    pf_trackers[i].invisible_count += 1
                    if is_in_boundary(pf_trackers[i].pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and pf_trackers[i].invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
                for c in avail_d:
                    if is_in_boundary(dets[c][:2]+dets[c][2:]/2, W, H, BOUNDARY_PERCENT):
                        pf_trackers.append(Tracker(next_tid, dets[c], device)); next_tid += 1
            elif pf_trackers:
                for i, t in enumerate(pf_trackers):
                    t.invisible_count += 1
                    if is_in_boundary(t.pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and t.invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
            pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in to_del]

        cv2.line(current_frame, (int(dp1[0]), int(dp1[1])), (int(dp2[0]), int(dp2[1])), (255, 0, 255), 2)
        for t in pf_trackers:
            if t.invisible_count == 0: t.pf.resample()
            s = t.pf.get_state().cpu().numpy()
            p1, p2 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2)), (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
            clr = (0, 255, 0) if t.invisible_count == 0 else (0, 0, 255)
            cv2.rectangle(current_frame, p1, p2, clr, 2)
            cv2.putText(current_frame, f"ID {t.id}", (p1[0], p1[1]-10), 0, 0.7, clr, 2)
        out_video.write(current_frame)

cap.release()
out_video.release()

In [ ]:
##Weighted Eucledian
### GPU-Optimized Repulsive Force Particle Filter (Weighted Anisotropic Distance)
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from transformers import DetrForObjectDetection, DetrImageProcessor
import supervision as sv
import torchvision
import math
import glob
import json

video_path = '/content/drive/MyDrive/croppedVideoForAnnotation.MP4'
HOME = "/content/drive/MyDrive/Updated DETR Model"
model_dir = os.path.join(HOME, 'custom-modelv2-50epochs')
output_directory = '/content/drive/MyDrive/ParticleFilter'
output_filename = "output_APF_GPU_Weighted_Distance.mp4"
output_video_path = os.path.join(output_directory, output_filename)
os.makedirs(output_directory, exist_ok=True)
COCO_ANNOTATIONS_FILE_PATH = '/content/drive/MyDrive/ParticleFilterWithPartition/train/_annotations.coco.json'

FRAME_ID_TO_LOAD = 0
FRAME_SKIP = 1
MAX_SECONDS_TO_PROCESS = 10
DIVIDER_P1 = (0, 104)
DIVIDER_P2 = (496.809, 377.316)
ROAD_ANGLE_NORTH = 28.8
ROAD_ANGLE_SOUTH = 208.8
CONFIDENCE_THRESHOLD = 0.95
NMS_IOU_THRESHOLD = 0.5
W_LONGITUDINAL = 1.0
W_LATERAL = 4.0
DISTANCE_MATCH_THRESHOLD = 150.0
MAX_INVISIBLE_COUNT = 6
BOUNDARY_PERCENT = 0.05
NUM_PARTICLES = 75
PROCESS_NOISE_STD = {'x': 2, 'y': 2, 'vx': 1.5, 'vy': 2, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 20
SIGMA_REPULSION = 50.0

def is_in_boundary(center_point_tensor, W, H, percent):
    bound_x, bound_y = W * percent, H * percent
    cx, cy = center_point_tensor[0], center_point_tensor[1]
    return (cx < bound_x) or (cx > W - bound_x) or (cy < bound_y) or (cy > H - bound_y)

def is_north_of_line(point_tensor, line_p1, line_p2):
    x, y = point_tensor[:, 0], point_tensor[:, 1]
    x1, y1 = line_p1; x2, y2 = line_p2
    return ((x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)) > 0

def calculate_weighted_distance_torch(track_centers, det_centers, track_angles_rad, w_long, w_lat):
    diff_x = track_centers[:, 0].unsqueeze(1) - det_centers[:, 0].unsqueeze(0)
    diff_y = track_centers[:, 1].unsqueeze(1) - det_centers[:, 1].unsqueeze(0)
    cos_a, sin_a = torch.cos(track_angles_rad).unsqueeze(1), torch.sin(track_angles_rad).unsqueeze(1)
    dist_long = diff_x * cos_a + diff_y * sin_a
    dist_lat = -diff_x * sin_a + diff_y * cos_a
    return torch.sqrt((dist_long * w_long)**2 + (dist_lat * w_lat)**2)

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device, self.num_particles, self.measurement_noise_std = device, num_particles, measurement_noise_std
        self.particles = torch.empty((num_particles, self.STATE_DIM), device=device)
        self.weights = torch.ones(num_particles, device=device) / num_particles
        self.proc_noise = torch.tensor([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=device).float()

    def initialize(self, bbox_tensor):
        cx, cy, w, h = bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2, bbox_tensor[2], bbox_tensor[3]
        initial = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float()
        self.particles = initial + torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self, other_states, ang_n, ang_s, p1, p2):
        pos = self.particles[:, 0:2]
        A_rep = torch.zeros((self.num_particles, 2), device=self.device)
        if other_states.numel() > 0:
            vecs = other_states[:, 0:2].unsqueeze(0) - pos.unsqueeze(1)
            dist = torch.clamp(torch.norm(vecs, dim=2), min=1e-6)
            force = torch.exp(-(dist**2) / (SIGMA_REPULSION**2))
            A_rep = -torch.sum(force.unsqueeze(2) * (vecs / dist.unsqueeze(2)), dim=1)
        self.particles[:, 2:4] += torch.nan_to_num(A_rep)
        self.particles[:, 0:2] += self.particles[:, 2:4]
        self.particles += torch.randn_like(self.particles) * self.proc_noise

    def update(self, bbox):
        det_c = torch.tensor([bbox[0] + bbox[2]/2, bbox[1] + bbox[3]/2], device=self.device)
        likelihood = torch.exp(-0.5 * (torch.norm(self.particles[:, 0:2] - det_c, dim=1) / self.measurement_noise_std)**2)
        self.weights = (self.weights * torch.nan_to_num(likelihood, nan=1e-10)) + 1e-300
        self.weights /= torch.sum(self.weights)

    def resample(self):
        self.weights = torch.nan_to_num(torch.clamp(self.weights, min=0.0), nan=1.0/self.num_particles)
        s = torch.sum(self.weights)
        w = self.weights / s if s > 0 else torch.ones_like(self.weights)/self.num_particles
        idx = torch.multinomial(w, self.num_particles, replacement=True)
        self.particles, self.weights = self.particles[idx], torch.ones(self.num_particles, device=self.device)/self.num_particles

    def get_state(self):
        w = torch.nan_to_num(self.weights, nan=0.0)
        p = torch.nan_to_num(self.particles, nan=0.0)
        s = torch.sum(w)
        return torch.sum(p * (w/s).unsqueeze(1), dim=0) if s > 0 else torch.mean(p, dim=0)

class Tracker:
    def __init__(self, tid, bbox, dev):
        self.id, self.invisible_count = tid, 0
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, dev)
        self.pf.initialize(bbox)

def run_detector(frame, model, proc, dev):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = proc(images=rgb, return_tensors='pt').to(dev)
        outputs = model(**inputs)
        res = proc.post_process_object_detection(outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=torch.tensor([frame.shape[:2]], device=dev))[0]
    if not res['scores'].numel(): return torch.empty((0, 4), device=dev)
    raw = sv.Detections.from_transformers(transformers_results=res)
    boxes, scores = torch.from_numpy(raw.xyxy).to(dev), torch.from_numpy(raw.confidence).to(dev)
    keep = torchvision.ops.nms(boxes, scores, iou_threshold=NMS_IOU_THRESHOLD)
    xy = raw.xyxy[keep.cpu().numpy()]
    return torch.from_numpy(np.stack([xy[:, 0], xy[:, 1], xy[:, 2]-xy[:, 0], xy[:, 3]-xy[:, 1]], axis=1)).float().to(dev)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained(model_dir, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(model_dir)
model.eval()

cap = cv2.VideoCapture(video_path)
W, H, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
total_f = min(int(cap.get(7)), int(MAX_SECONDS_TO_PROCESS * fps))
out_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/FRAME_SKIP, (W, H))

ang_n, ang_s = torch.deg2rad(torch.tensor(ROAD_ANGLE_NORTH, device=device)), torch.deg2rad(torch.tensor(ROAD_ANGLE_SOUTH, device=device))
dp1, dp2 = torch.tensor(DIVIDER_P1, device=device).float(), torch.tensor(DIVIDER_P2, device=device).float()
pf_trackers, next_tid = [], 0

for fidx in tqdm(range(total_f)):
    ret, frame = cap.read()
    if not ret: break
    if fidx % FRAME_SKIP == 0:
        current_frame = frame.copy()
        if fidx == 0:
            with open(COCO_ANNOTATIONS_FILE_PATH, 'r') as f:
                anns = [a['bbox'] for a in json.load(f).get('annotations', []) if a.get('image_id') == FRAME_ID_TO_LOAD]
            dets = torch.tensor(anns, dtype=torch.float32).to(device)
            for i in range(dets.shape[0]):
                pf_trackers.append(Tracker(next_tid, dets[i], device)); next_tid += 1
        else:
            dets = run_detector(current_frame, model, processor, device)
            if pf_trackers:
                st_all = torch.stack([t.pf.get_state() for t in pf_trackers])
                for i, t in enumerate(pf_trackers):
                    others = st_all[[j for j in range(len(pf_trackers)) if i != j]] if len(pf_trackers) > 1 else torch.empty((0, 6), device=device)
                    t.pf.predict(others, ang_n, ang_s, dp1, dp2)

            to_del = set()
            if pf_trackers and dets.numel() > 0:
                st = torch.stack([t.pf.get_state() for t in pf_trackers])
                is_n = is_north_of_line(st[:, :2], dp1, dp2)
                cost = calculate_weighted_distance_torch(st[:, :2], dets[:, :2] + dets[:, 2:]/2.0, torch.where(is_n, ang_n, ang_s), W_LONGITUDINAL, W_LATERAL).cpu().numpy()
                t_idx, d_idx = np.unravel_index(np.argsort(cost, axis=None), cost.shape)
                matched, avail_d, used_t = set(), set(range(dets.shape[0])), set()
                for r, c in zip(t_idx, d_idx):
                    if r not in used_t and c in avail_d and cost[r, c] < DISTANCE_MATCH_THRESHOLD:
                        pf_trackers[r].pf.update(dets[c]); pf_trackers[r].invisible_count = 0
                        matched.add(r); used_t.add(r); avail_d.remove(c)
                for i in (set(range(len(pf_trackers))) - matched):
                    pf_trackers[i].invisible_count += 1
                    if is_in_boundary(pf_trackers[i].pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and pf_trackers[i].invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
                for c in avail_d:
                    if is_in_boundary(dets[c][:2]+dets[c][2:]/2, W, H, BOUNDARY_PERCENT):
                        pf_trackers.append(Tracker(next_tid, dets[c], device)); next_tid += 1
            elif pf_trackers:
                for i, t in enumerate(pf_trackers):
                    t.invisible_count += 1
                    if is_in_boundary(t.pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and t.invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
            pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in to_del]

        cv2.line(current_frame, (int(dp1[0]), int(dp1[1])), (int(dp2[0]), int(dp2[1])), (255, 0, 255), 2)
        for t in pf_trackers:
            if t.invisible_count == 0: t.pf.resample()
            s = t.pf.get_state().cpu().numpy()
            p1, p2 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2)), (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
            clr = (0, 255, 0) if t.invisible_count == 0 else (0, 0, 255)
            cv2.rectangle(current_frame, p1, p2, clr, 2)
            cv2.putText(current_frame, f"ID {t.id}", (p1[0], p1[1]-10), 0, 0.7, clr, 2)
        out_video.write(current_frame)

cap.release()
out_video.release()

In [ ]:
### GPU-Optimized Repulsive Force Particle Filter (Steering Physics APF  + Blended Matching of IOU + Eucledian )
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from transformers import DetrForObjectDetection, DetrImageProcessor
from scipy.optimize import linear_sum_assignment
import supervision as sv
import torchvision
import math
import glob
import json

video_path = '/content/drive/MyDrive/croppedVideoForAnnotation.MP4'
HOME = "/content/drive/MyDrive/Updated DETR Model"
model_dir = os.path.join(HOME, 'custom-modelv2-50epochs')
output_directory = '/content/drive/MyDrive/ParticleFilter'
output_filename = "output_APF_GPU_Steering_Physics.mp4"
output_video_path = os.path.join(output_directory, output_filename)
os.makedirs(output_directory, exist_ok=True)
COCO_ANNOTATIONS_FILE_PATH = '/content/drive/MyDrive/ParticleFilterWithPartition/train/_annotations.coco.json'

FRAME_ID_TO_LOAD = 0
FRAME_SKIP = 1
MAX_SECONDS_TO_PROCESS = 10
DIVIDER_P1 = (0, 104)
DIVIDER_P2 = (496.809, 377.316)
ROAD_ANGLE_NORTH = 28.8
ROAD_ANGLE_SOUTH = 208.8
CONFIDENCE_THRESHOLD = 0.85
NMS_IOU_THRESHOLD = 0.5
MATCHING_ALPHA = 0.4
MATCHING_THRESHOLD = 0.8
MAX_INVISIBLE_COUNT = 6
BOUNDARY_PERCENT = 0.05
NUM_PARTICLES = 100
PROCESS_NOISE_STD = {'x': 1.0, 'y': 1.0, 'vx': 1.0, 'vy': 1.0, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 3
SIGMA_REPULSION = 40.0

def is_in_boundary(center_point_tensor, W, H, percent):
    bound_x, bound_y = W * percent, H * percent
    cx, cy = center_point_tensor[0], center_point_tensor[1]
    return (cx < bound_x) or (cx > W - bound_x) or (cy < bound_y) or (cy > H - bound_y)

def is_north_of_line(point_tensor, line_p1, line_p2):
    x, y = point_tensor[:, 0], point_tensor[:, 1]
    x1, y1 = line_p1; x2, y2 = line_p2
    return ((x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)) > 0

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device = device; self.num_particles = num_particles; self.measurement_noise_std = measurement_noise_std
        self.particles = torch.empty((num_particles, self.STATE_DIM), device=device)
        self.weights = torch.ones(num_particles, device=device) / num_particles
        self.proc_noise = torch.tensor([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=device).float()

    def initialize(self, bbox_tensor):
        cx, cy, w, h = bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2, bbox_tensor[2], bbox_tensor[3]
        initial = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float()
        self.particles = initial + torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self, other_states, ang_n, ang_s, p1, p2):
        pos, vel = self.particles[:, 0:2], self.particles[:, 2:4]
        angs = torch.where(is_north_of_line(pos, p1, p2), ang_n, ang_s)
        speed = torch.clamp(torch.norm(vel, dim=1).unsqueeze(1), min=0.1)
        target_s = torch.where(speed < 0.1, 0.5, speed)
        V_des = torch.cat([target_s * torch.cos(angs).unsqueeze(1), target_s * torch.sin(angs).unsqueeze(1)], dim=1)
        A_des = V_des - vel
        A_rep = torch.zeros((self.num_particles, 2), device=self.device)
        if other_states.numel() > 0:
            vecs = other_states[:, 0:2].unsqueeze(0) - pos.unsqueeze(1)
            dist = torch.clamp(torch.norm(vecs, dim=2), min=1e-6)
            force = torch.exp(-(dist**2) / (SIGMA_REPULSION**2))
            A_rep = -torch.sum(force.unsqueeze(2) * (vecs / dist.unsqueeze(2)), dim=1)
        accel = torch.nan_to_num(A_des + A_rep)
        self.particles[:, 2:4] += accel
        self.particles[:, 0:2] += self.particles[:, 2:4]
        self.particles += torch.randn_like(self.particles) * self.proc_noise

    def update(self, bbox):
        det_c = torch.tensor([bbox[0] + bbox[2]/2, bbox[1] + bbox[3]/2], device=self.device)
        likelihood = torch.exp(-0.5 * (torch.norm(self.particles[:, 0:2] - det_c, dim=1) / self.measurement_noise_std)**2)
        self.weights = (self.weights * torch.nan_to_num(likelihood, nan=1e-10)) + 1e-300
        self.weights /= torch.sum(self.weights)

    def resample(self):
        self.weights = torch.nan_to_num(torch.clamp(self.weights, min=0.0), nan=1.0/self.num_particles)
        s = torch.sum(self.weights)
        w = self.weights / s if s > 0 else torch.ones_like(self.weights)/self.num_particles
        idx = torch.multinomial(w, self.num_particles, replacement=True)
        self.particles, self.weights = self.particles[idx], torch.ones(self.num_particles, device=self.device)/self.num_particles

    def get_state(self):
        w = torch.nan_to_num(self.weights, nan=0.0)
        p = torch.nan_to_num(self.particles, nan=0.0)
        s = torch.sum(w)
        return torch.sum(p * (w/s).unsqueeze(1), dim=0) if s > 0 else torch.mean(p, dim=0)

class Tracker:
    def __init__(self, tid, bbox, dev):
        self.id, self.invisible_count = tid, 0
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, dev)
        self.pf.initialize(bbox)

def calculate_iou_torch(b1, b2):
    x1y1_1, x2y2_1 = b1[:, :2], b1[:, :2] + b1[:, 2:]
    x1y1_2, x2y2_2 = b2[:, :2], b2[:, :2] + b2[:, 2:]
    xi1, yi1 = torch.max(x1y1_1[:, 0].unsqueeze(1), x1y1_2[:, 0].unsqueeze(0)), torch.max(x1y1_1[:, 1].unsqueeze(1), x1y1_2[:, 1].unsqueeze(0))
    xi2, yi2 = torch.min(x2y2_1[:, 0].unsqueeze(1), x2y2_2[:, 0].unsqueeze(0)), torch.min(x2y2_1[:, 1].unsqueeze(1), x2y2_2[:, 1].unsqueeze(0))
    inter = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    union = (b1[:, 2] * b1[:, 3]).unsqueeze(1) + (b2[:, 2] * b2[:, 3]).unsqueeze(0) - inter
    return torch.nan_to_num(inter / (union + 1e-6), nan=0.0)

def run_detector(frame, model, proc, dev):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = proc(images=rgb, return_tensors='pt').to(dev)
        outputs = model(**inputs)
        res = proc.post_process_object_detection(outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=torch.tensor([frame.shape[:2]], device=dev))[0]
    if not res['scores'].numel(): return torch.empty((0, 4), device=dev)
    raw = sv.Detections.from_transformers(transformers_results=res)
    boxes, scores = torch.from_numpy(raw.xyxy).to(dev), torch.from_numpy(raw.confidence).to(dev)
    keep = torchvision.ops.nms(boxes, scores, NMS_IOU_THRESHOLD)
    xy = raw.xyxy[keep.cpu().numpy()]
    return torch.from_numpy(np.stack([xy[:, 0], xy[:, 1], xy[:, 2]-xy[:, 0], xy[:, 3]-xy[:, 1]], axis=1)).float().to(dev)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained(model_dir, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(model_dir)
model.eval()

cap = cv2.VideoCapture(video_path)
W, H, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
total_f = min(int(cap.get(7)), int(MAX_SECONDS_TO_PROCESS * fps))
out_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/FRAME_SKIP, (W, H))

ang_n, ang_s = torch.deg2rad(torch.tensor(ROAD_ANGLE_NORTH, device=device)), torch.deg2rad(torch.tensor(ROAD_ANGLE_SOUTH, device=device))
dp1, dp2 = torch.tensor(DIVIDER_P1, device=device).float(), torch.tensor(DIVIDER_P2, device=device).float()
pf_trackers, next_tid = [], 0

for fidx in tqdm(range(total_f)):
    ret, frame = cap.read()
    if not ret: break
    if fidx % FRAME_SKIP == 0:
        if fidx == 0:
            with open(COCO_ANNOTATIONS_FILE_PATH, 'r') as f:
                anns = [a['bbox'] for a in json.load(f).get('annotations', []) if a.get('image_id') == FRAME_ID_TO_LOAD and a.get('id') != 9]
            dets = torch.tensor(anns, dtype=torch.float32).to(device)
            for i in range(dets.shape[0]):
                pf_trackers.append(Tracker(next_tid, dets[i], device)); next_tid += 1
        else:
            dets = run_detector(frame, model, processor, device)
            if pf_trackers:
                st_all = torch.stack([t.pf.get_state() for t in pf_trackers])
                for i, t in enumerate(pf_trackers):
                    others = st_all[[j for j in range(len(pf_trackers)) if i != j]] if len(pf_trackers) > 1 else torch.empty((0, 6), device=device)
                    t.pf.predict(others, ang_n, ang_s, dp1, dp2)

            to_del = set()
            if pf_trackers and dets.numel() > 0:
                st = torch.stack([t.pf.get_state() for t in pf_trackers])
                t_b = torch.stack([st[:,0]-st[:,4]/2, st[:,1]-st[:,5]/2, st[:,4], st[:,5]], axis=1)
                cost = (MATCHING_ALPHA * (1.0 - calculate_iou_torch(t_b, dets))) + \
                       ((1-MATCHING_ALPHA) * (torch.norm(st[:,:2].unsqueeze(1)-(dets[:,:2]+dets[:,2:]/2).unsqueeze(0), dim=2)/(torch.tensor([W,H], device=device).norm()+1e-6)))
                r_idx, c_idx = linear_sum_assignment(cost.cpu().numpy())
                matched, avail_d = set(), set(range(dets.shape[0]))
                for r, c in zip(r_idx, c_idx):
                    if cost[r, c] < MATCHING_THRESHOLD:
                        pf_trackers[r].pf.update(dets[c]); pf_trackers[r].invisible_count = 0
                        matched.add(r); avail_d.discard(c)
                for i in (set(range(len(pf_trackers))) - matched):
                    pf_trackers[i].invisible_count += 1
                    if is_in_boundary(pf_trackers[i].pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and pf_trackers[i].invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
                for c in avail_d:
                    if is_in_boundary(dets[c][:2]+dets[c][2:]/2, W, H, BOUNDARY_PERCENT):
                        pf_trackers.append(Tracker(next_tid, dets[c], device)); next_tid += 1
            elif pf_trackers:
                for i, t in enumerate(pf_trackers):
                    t.invisible_count += 1
                    if is_in_boundary(t.pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and t.invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
            pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in to_del]

        cv2.line(frame, (int(dp1[0]), int(dp1[1])), (int(dp2[0]), int(dp2[1])), (255, 0, 255), 2)
        for t in pf_trackers:
            if t.invisible_count == 0: t.pf.resample()
            s = t.pf.get_state().cpu().numpy()
            p1, p2 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2)), (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
            clr = (0, 255, 0) if t.invisible_count == 0 else (0, 0, 255)
            cv2.rectangle(frame, p1, p2, clr, 2)
            cv2.putText(frame, f"ID {t.id}", (p1[0], p1[1]-10), 0, 0.7, clr, 2)
        out_video.write(frame)

cap.release()
out_video.release()

In [ ]:
### GPU-Optimized Repulsive Force Particle Filter (APF Steering Physics + Divider Constraint)
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from transformers import DetrForObjectDetection, DetrImageProcessor
from scipy.optimize import linear_sum_assignment
import supervision as sv
import torchvision
import math
import glob
import json

video_path = '/content/drive/MyDrive/croppedVideoForAnnotation.MP4'
HOME = "/content/drive/MyDrive/Updated DETR Model"
model_dir = os.path.join(HOME, 'custom-modelv2-50epochs')
output_directory = '/content/drive/MyDrive/ParticleFilter'
output_filename = "output_APF_GPU_Divider_Constraint.mp4"
output_video_path = os.path.join(output_directory, output_filename)
os.makedirs(output_directory, exist_ok=True)
COCO_ANNOTATIONS_FILE_PATH = '/content/drive/MyDrive/ParticleFilterWithPartition/train/_annotations.coco.json'

FRAME_ID_TO_LOAD = 0
FRAME_SKIP = 1
MAX_SECONDS_TO_PROCESS = 10
DIVIDER_P1 = (0, 104)
DIVIDER_P2 = (496.809, 377.316)
ROAD_ANGLE_NORTH = 28.8
ROAD_ANGLE_SOUTH = 208.8
CONFIDENCE_THRESHOLD = 0.85
NMS_IOU_THRESHOLD = 0.5
MATCHING_ALPHA = 0.4
MATCHING_THRESHOLD = 0.9
MAX_INVISIBLE_COUNT = 6
BOUNDARY_PERCENT = 0.05
NUM_PARTICLES = 100
PROCESS_NOISE_STD = {'x': 2.0, 'y': 2.0, 'vx': 1.0, 'vy': 1.0, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 5
SIGMA_REPULSION = 40.0

def is_in_boundary(center_point_tensor, W, H, percent):
    bound_x, bound_y = W * percent, H * percent
    cx, cy = center_point_tensor[0], center_point_tensor[1]
    return (cx < bound_x) or (cx > W - bound_x) or (cy < bound_y) or (cy > H - bound_y)

def is_north_of_line(point_tensor, line_p1, line_p2):
    x, y = point_tensor[:, 0], point_tensor[:, 1]
    x1, y1 = line_p1; x2, y2 = line_p2
    return ((x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)) > 0

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device = device; self.num_particles = num_particles; self.measurement_noise_std = measurement_noise_std
        self.particles = torch.empty((self.num_particles, self.STATE_DIM), device=self.device)
        self.weights = torch.ones(self.num_particles, device=self.device) / self.num_particles
        self.proc_noise = torch.tensor([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=self.device).float()

    def initialize(self, bbox_tensor):
        cx, cy, w, h = bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2, bbox_tensor[2], bbox_tensor[3]
        initial_state = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float()
        self.particles = initial_state + torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self, other_vehicle_states_tensor, road_angle_north_rad, road_angle_south_rad, divider_p1, divider_p2):
        particle_pos, particle_vel = self.particles[:, 0:2], self.particles[:, 2:4]
        is_north = is_north_of_line(particle_pos, divider_p1, divider_p2)
        road_angles = torch.where(is_north, road_angle_north_rad, road_angle_south_rad)
        current_speed = torch.norm(particle_vel, dim=1).unsqueeze(1)
        speed_mag = torch.where(current_speed < 0.1, 2.0, current_speed)
        V_desired = torch.cat([speed_mag * torch.cos(road_angles).unsqueeze(1), speed_mag * torch.sin(road_angles).unsqueeze(1)], dim=1)
        A_desired = V_desired - particle_vel
        A_repulsive = torch.zeros((self.num_particles, 2), device=self.device)
        if other_vehicle_states_tensor.numel() > 0:
            vec_to_other = other_vehicle_states_tensor[:, 0:2].unsqueeze(0) - particle_pos.unsqueeze(1)
            dist = torch.clamp(torch.norm(vec_to_other, dim=2), min=1e-6)
            force_mag = torch.exp(-(dist**2) / (SIGMA_REPULSION**2))
            A_repulsive = -torch.sum(force_mag.unsqueeze(2) * (vec_to_other / dist.unsqueeze(2)), dim=1)
        resultant_accel = A_desired + A_repulsive
        self.particles[:, 2:4] += torch.nan_to_num(resultant_accel)
        self.particles[:, 0:2] += self.particles[:, 2:4]
        self.particles += torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def update(self, bbox_tensor):
        det_c = torch.tensor([bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2], device=self.device)
        likelihood = torch.exp(-0.5 * (torch.norm(self.particles[:, 0:2] - det_c, dim=1) / self.measurement_noise_std)**2)
        self.weights = (self.weights * torch.nan_to_num(likelihood, nan=1e-10)) + 1e-300
        self.weights /= torch.sum(self.weights)

    def resample(self):
        self.weights = torch.nan_to_num(torch.clamp(self.weights, min=0.0), nan=1.0/self.num_particles)
        sum_w = torch.sum(self.weights)
        w = self.weights / sum_w if sum_w > 0 else torch.ones_like(self.weights)/self.num_particles
        indices = torch.multinomial(w, self.num_particles, replacement=True)
        self.particles, self.weights = self.particles[indices], torch.ones(self.num_particles, device=self.device)/self.num_particles

    def get_state(self):
        w = torch.nan_to_num(self.weights, nan=0.0)
        p = torch.nan_to_num(self.particles, nan=0.0)
        s = torch.sum(w)
        return torch.sum(p * (w/s).unsqueeze(1), dim=0) if s > 0 else torch.mean(p, dim=0)

class Tracker:
    def __init__(self, track_id, initial_bbox_tensor, device, is_north_init):
        self.id, self.invisible_count = track_id, 0
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, device)
        self.pf.initialize(initial_bbox_tensor)
        self.original_side_north = is_north_init

def calculate_iou_torch(boxes1, boxes2):
    x1y1_1, x2y2_1 = boxes1[:, :2], boxes1[:, :2] + boxes1[:, 2:]
    x1y1_2, x2y2_2 = boxes2[:, :2], boxes2[:, :2] + boxes2[:, 2:]
    xi1, yi1 = torch.max(x1y1_1[:, 0].unsqueeze(1), x1y1_2[:, 0].unsqueeze(0)), torch.max(x1y1_1[:, 1].unsqueeze(1), x1y1_2[:, 1].unsqueeze(0))
    xi2, yi2 = torch.min(x2y2_1[:, 0].unsqueeze(1), x2y2_2[:, 0].unsqueeze(0)), torch.min(x2y2_1[:, 1].unsqueeze(1), x2y2_2[:, 1].unsqueeze(0))
    inter = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    union = (boxes1[:, 2] * boxes1[:, 3]).unsqueeze(1) + (boxes2[:, 2] * boxes2[:, 3]).unsqueeze(0) - inter
    return torch.nan_to_num(inter / (union + 1e-6), nan=0.0)

def calculate_distance_torch(centers1, centers2):
    return torch.nan_to_num(torch.norm(centers1.unsqueeze(1) - centers2.unsqueeze(0), dim=2), nan=float('inf'))

def run_detector(frame_bgr, model, image_processor, device):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = image_processor(images=rgb, return_tensors='pt').to(device)
        outputs = model(**inputs)
        res = image_processor.post_process_object_detection(outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=torch.tensor([frame_bgr.shape[:2]], device=device))[0]
    if len(res['scores']) == 0: return torch.empty((0, 4), device=device)
    raw = sv.Detections.from_transformers(transformers_results=res)
    boxes, scores = torch.from_numpy(raw.xyxy).to(device), torch.from_numpy(raw.confidence).to(device)
    keep = torchvision.ops.nms(boxes, scores, iou_threshold=NMS_IOU_THRESHOLD)
    xyxy = raw.xyxy[keep.cpu().numpy()]
    return torch.from_numpy(np.stack([xyxy[:, 0], xyxy[:, 1], xyxy[:, 2]-xyxy[:, 0], xyxy[:, 3]-xyxy[:, 1]], axis=1)).float().to(device)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained(model_dir, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(model_dir)
model.eval()

cap = cv2.VideoCapture(video_path)
W, H, source_fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
total_frames = min(int(cap.get(7)), int(MAX_SECONDS_TO_PROCESS * source_fps))
out_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), source_fps/FRAME_SKIP, (W, H))

ang_n, ang_s = torch.deg2rad(torch.tensor(ROAD_ANGLE_NORTH, device=device)), torch.deg2rad(torch.tensor(ROAD_ANGLE_SOUTH, device=device))
dp1, dp2 = torch.tensor(DIVIDER_P1, device=device).float(), torch.tensor(DIVIDER_P2, device=device).float()
pf_trackers, next_track_id = [], 0

for fidx in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret: break
    if fidx % FRAME_SKIP == 0:
        if fidx == 0:
            with open(COCO_ANNOTATIONS_FILE_PATH, 'r') as f:
                anns = [a['bbox'] for a in json.load(f).get('annotations', []) if a.get('image_id') == FRAME_ID_TO_LOAD and a.get('id') != 9]
            detections_tensor = torch.tensor(anns, dtype=torch.float32).to(device)
            if detections_tensor.numel() > 0:
                is_north_tensor = is_north_of_line(detections_tensor[:, :2] + detections_tensor[:, 2:] / 2.0, dp1, dp2)
                for i in range(detections_tensor.shape[0]):
                    pf_trackers.append(Tracker(next_track_id, detections_tensor[i], device, is_north_tensor[i].item()))
                    next_track_id += 1
        else:
            detections_tensor = run_detector(frame, model, processor, device)
            if pf_trackers:
                states = torch.stack([t.pf.get_state() for t in pf_trackers])
                for i, tracker in enumerate(pf_trackers):
                    others = states[[j for j in range(len(pf_trackers)) if i != j]] if len(pf_trackers) > 1 else torch.empty((0, 6), device=device)
                    tracker.pf.predict(others, ang_n, ang_s, dp1, dp2)

            to_del = set()
            if pf_trackers and detections_tensor.numel() > 0:
                st = torch.stack([t.pf.get_state() for t in pf_trackers])
                t_b = torch.stack([st[:,0]-st[:,4]/2, st[:,1]-st[:,5]/2, st[:,4], st[:,5]], axis=1)
                cost = (MATCHING_ALPHA * (1.0 - calculate_iou_torch(t_b, detections_tensor))) + \
                       ((1 - MATCHING_ALPHA) * (calculate_distance_torch(st[:, :2], detections_tensor[:, :2] + detections_tensor[:, 2:] / 2.0) / (torch.tensor([W, H], device=device).norm() + 1e-6)))

                track_sides = torch.tensor([t.original_side_north for t in pf_trackers], device=device)
                det_sides = is_north_of_line(detections_tensor[:, :2] + detections_tensor[:, 2:] / 2.0, dp1, dp2)
                cost[track_sides.unsqueeze(1) ^ det_sides.unsqueeze(0)] = 10000.0

                r_idx, c_idx = linear_sum_assignment(cost.cpu().numpy())
                matched, avail_d = set(), set(range(detections_tensor.shape[0]))
                for r, c in zip(r_idx, c_idx):
                    if cost[r, c] < MATCHING_THRESHOLD:
                        pf_trackers[r].pf.update(detections_tensor[c]); pf_trackers[r].invisible_count = 0
                        matched.add(r); avail_d.discard(c)
                for i in (set(range(len(pf_trackers))) - matched):
                    pf_trackers[i].invisible_count += 1
                    if is_in_boundary(pf_trackers[i].pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and pf_trackers[i].invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
                for c in avail_d:
                    if is_in_boundary(detections_tensor[c][:2] + detections_tensor[c][2:]/2, W, H, BOUNDARY_PERCENT):
                        pf_trackers.append(Tracker(next_track_id, detections_tensor[c], device, det_sides[c].item())); next_track_id += 1
            elif pf_trackers:
                for i, t in enumerate(pf_trackers):
                    t.invisible_count += 1
                    if is_in_boundary(t.pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and t.invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
            pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in to_del]

        cv2.line(frame, (int(dp1[0]), int(dp1[1])), (int(dp2[0]), int(dp2[1])), (255, 0, 255), 2)
        for t in pf_trackers:
            if t.invisible_count == 0: t.pf.resample()
            s = t.pf.get_state().cpu().numpy()
            p1, p2 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2)), (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
            clr = (0, 255, 0) if t.invisible_count == 0 else (0, 0, 255)
            cv2.rectangle(frame, p1, p2, clr, 2)
            cv2.putText(frame, f"ID {t.id}", (p1[0], p1[1]-10), 0, 0.7, clr, 2)
        out_video.write(frame)

cap.release()
out_video.release()

In [ ]:
### HYBRID GPU PARTICLE FILTER

# Motion: Velocity Blending

# Association: Hungarian Algorithm + Blended Cost
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from transformers import DetrForObjectDetection, DetrImageProcessor
from scipy.optimize import linear_sum_assignment
import supervision as sv
import torchvision
import math
import glob
import json

video_path = '/content/drive/MyDrive/croppedVideoForAnnotation.MP4'
HOME = "/content/drive/MyDrive/Updated DETR Model"
model_dir = os.path.join(HOME, 'custom-modelv2-50epochs')
output_directory = '/content/drive/MyDrive/ParticleFilter'
output_filename = "output_APF_Hybrid_BestOfBoth.mp4"
output_video_path = os.path.join(output_directory, output_filename)
os.makedirs(output_directory, exist_ok=True)
COCO_ANNOTATIONS_FILE_PATH = '/content/drive/MyDrive/ParticleFilterWithPartition/train/_annotations.coco.json'

FRAME_ID_TO_LOAD = 0
FRAME_SKIP = 1
MAX_SECONDS_TO_PROCESS = 10
DIVIDER_P1 = (0, 104)
DIVIDER_P2 = (496.809, 377.316)
ROAD_ANGLE_NORTH = 28.8
ROAD_ANGLE_SOUTH = 208.8
CONFIDENCE_THRESHOLD = 0.85
NMS_IOU_THRESHOLD = 0.5
MATCHING_ALPHA = 0.4
MATCHING_THRESHOLD = 0.8
MAX_INVISIBLE_COUNT = 6
BOUNDARY_PERCENT = 0.05
NUM_PARTICLES = 150
PROCESS_NOISE_STD = {'x': 1.0, 'y': 1.0, 'vx': 1.0, 'vy': 1.0, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 3
SIGMA_REPULSION = 40.0

def is_in_boundary(center_point_tensor, W, H, percent):
    bound_x, bound_y = W * percent, H * percent
    cx, cy = center_point_tensor[0], center_point_tensor[1]
    return (cx < bound_x) or (cx > W - bound_x) or (cy < bound_y) or (cy > H - bound_y)

def is_north_of_line(point_tensor, line_p1, line_p2):
    x, y = point_tensor[:, 0], point_tensor[:, 1]
    x1, y1 = line_p1; x2, y2 = line_p2
    return ((x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)) > 0

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device = device; self.num_particles = num_particles; self.measurement_noise_std = measurement_noise_std
        self.particles = torch.empty((self.num_particles, self.STATE_DIM), device=self.device)
        self.weights = torch.ones(self.num_particles, device=self.device) / self.num_particles
        self.proc_noise = torch.tensor([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=self.device).float()

    def initialize(self, bbox_tensor):
        cx, cy, w, h = bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2, bbox_tensor[2], bbox_tensor[3]
        self.particles = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float() + \
                         torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self, other_vehicle_states_tensor, road_angle_north_rad, road_angle_south_rad, divider_p1, divider_p2):
        particle_pos = self.particles[:, 0:2]
        is_north = is_north_of_line(particle_pos, divider_p1, divider_p2)
        road_angles = torch.where(is_north, road_angle_north_rad, road_angle_south_rad)
        current_speed = torch.norm(self.particles[:, 2:4], dim=1)
        target_speed = torch.where(current_speed < 0.5, torch.tensor(0.0, device=self.device), torch.tensor(2.0, device=self.device))
        V_desired = torch.stack([target_speed * torch.cos(road_angles), target_speed * torch.sin(road_angles)], dim=1)
        V_repulsive = torch.zeros((self.num_particles, 2), device=self.device)
        if other_vehicle_states_tensor.numel() > 0:
            vec_to_other = other_vehicle_states_tensor[:, 0:2].unsqueeze(0) - particle_pos.unsqueeze(1)
            distances = torch.clamp(torch.norm(vec_to_other, dim=2), min=1e-1)
            force_mag = torch.exp(-(distances**2) / (SIGMA_REPULSION**2))
            V_repulsive = -torch.sum(force_mag.unsqueeze(2) * (vec_to_other / distances.unsqueeze(2)), dim=1)
        target_velocity = V_desired + V_repulsive
        alpha = 0.3
        self.particles[:, 2:4] = (1 - alpha) * self.particles[:, 2:4] + alpha * torch.nan_to_num(target_velocity)
        self.particles[:, 0:2] += self.particles[:, 2:4]
        self.particles += torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def update(self, bbox_tensor):
        det_c = torch.tensor([bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2], device=self.device)
        likelihood = torch.exp(-0.5 * (torch.norm(self.particles[:, 0:2] - det_c, dim=1) / self.measurement_noise_std)**2)
        self.weights = (self.weights * torch.nan_to_num(likelihood, nan=1e-10)) + 1e-300
        self.weights /= torch.sum(self.weights)

    def resample(self):
        self.weights = torch.nan_to_num(torch.clamp(self.weights, min=0.0), nan=1.0/self.num_particles)
        sum_w = torch.sum(self.weights)
        w = self.weights / sum_w if sum_w > 0 else torch.ones_like(self.weights)/self.num_particles
        indices = torch.multinomial(w, self.num_particles, replacement=True)
        self.particles, self.weights = self.particles[indices], torch.ones(self.num_particles, device=self.device)/self.num_particles

    def get_state(self):
        w = torch.nan_to_num(self.weights, nan=0.0)
        p = torch.nan_to_num(self.particles, nan=0.0)
        s = torch.sum(w)
        return torch.sum(p * (w/s).unsqueeze(1), dim=0) if s > 0 else torch.mean(p, dim=0)

class Tracker:
    def __init__(self, track_id, initial_bbox_tensor, device):
        self.id, self.invisible_count = track_id, 0
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, device)
        self.pf.initialize(initial_bbox_tensor)

def calculate_iou_torch(boxes1, boxes2):
    x1y1_1, x2y2_1 = boxes1[:, :2], boxes1[:, :2] + boxes1[:, 2:]
    x1y1_2, x2y2_2 = boxes2[:, :2], boxes2[:, :2] + boxes2[:, 2:]
    xi1, yi1 = torch.max(x1y1_1[:, 0].unsqueeze(1), x1y1_2[:, 0].unsqueeze(0)), torch.max(x1y1_1[:, 1].unsqueeze(1), x1y1_2[:, 1].unsqueeze(0))
    xi2, yi2 = torch.min(x2y2_1[:, 0].unsqueeze(1), x2y2_2[:, 0].unsqueeze(0)), torch.min(x2y2_1[:, 1].unsqueeze(1), x2y2_2[:, 1].unsqueeze(0))
    inter = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    union = (boxes1[:, 2] * boxes1[:, 3]).unsqueeze(1) + (boxes2[:, 2] * boxes2[:, 3]).unsqueeze(0) - inter
    return torch.nan_to_num(inter / (union + 1e-6), nan=0.0)

def run_detector(frame_bgr, model, image_processor, device):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = image_processor(images=rgb, return_tensors='pt').to(device)
        outputs = model(**inputs)
        res = image_processor.post_process_object_detection(outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=torch.tensor([frame_bgr.shape[:2]], device=device))[0]
    if len(res['scores']) == 0: return torch.empty((0, 4), device=device)
    raw = sv.Detections.from_transformers(transformers_results=res)
    boxes, scores = torch.from_numpy(raw.xyxy).to(device), torch.from_numpy(raw.confidence).to(device)
    keep = torchvision.ops.nms(boxes, scores, iou_threshold=NMS_IOU_THRESHOLD)
    xyxy = raw.xyxy[keep.cpu().numpy()]
    return torch.from_numpy(np.stack([xyxy[:, 0], xyxy[:, 1], xyxy[:, 2]-xyxy[:, 0], xyxy[:, 3]-xyxy[:, 1]], axis=1)).float().to(device)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DetrForObjectDetection.from_pretrained(model_dir, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(model_dir)
model.eval()

cap = cv2.VideoCapture(video_path)
W, H, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
total_frames = min(int(cap.get(7)), int(MAX_SECONDS_TO_PROCESS * fps))
out_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps/FRAME_SKIP, (W, H))

ang_n, ang_s = torch.deg2rad(torch.tensor(ROAD_ANGLE_NORTH, device=device)), torch.deg2rad(torch.tensor(ROAD_ANGLE_SOUTH, device=device))
dp1, dp2 = torch.tensor(DIVIDER_P1, device=device).float(), torch.tensor(DIVIDER_P2, device=device).float()
pf_trackers, next_track_id = [], 0

for fidx in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret: break
    if fidx % FRAME_SKIP == 0:
        if fidx == 0:
            with open(COCO_ANNOTATIONS_FILE_PATH, 'r') as f:
                anns = [a['bbox'] for a in json.load(f).get('annotations', []) if a.get('image_id') == FRAME_ID_TO_LOAD and a.get('id') != 9]
            dets = torch.tensor(anns, dtype=torch.float32).to(device)
            for i in range(dets.shape[0]):
                pf_trackers.append(Tracker(next_track_id, dets[i], device)); next_track_id += 1
        else:
            dets = run_detector(frame, model, processor, device)
            if pf_trackers:
                states = torch.stack([t.pf.get_state() for t in pf_trackers])
                for i, tracker in enumerate(pf_trackers):
                    others = states[[j for j in range(len(pf_trackers)) if i != j]] if len(pf_trackers)>1 else torch.empty((0, 6), device=device)
                    tracker.pf.predict(others, ang_n, ang_s, dp1, dp2)

            to_del = set()
            if pf_trackers and dets.numel() > 0:
                st = torch.stack([t.pf.get_state() for t in pf_trackers])
                t_b = torch.stack([st[:,0]-st[:,4]/2, st[:,1]-st[:,5]/2, st[:,4], st[:,5]], axis=1)
                cost = (MATCHING_ALPHA * (1.0 - calculate_iou_torch(t_b, dets))) + \
                       ((1-MATCHING_ALPHA) * (torch.norm(st[:,:2].unsqueeze(1)-(dets[:,:2]+dets[:,2:]/2).unsqueeze(0), dim=2)/(torch.tensor([W, H], device=device).norm()+1e-6)))
                r_idx, c_idx = linear_sum_assignment(cost.cpu().numpy())
                matched, avail_d = set(), set(range(dets.shape[0]))
                for r, c in zip(r_idx, c_idx):
                    if cost[r, c] < MATCHING_THRESHOLD:
                        pf_trackers[r].pf.update(dets[c]); pf_trackers[r].invisible_count = 0
                        matched.add(r); avail_d.discard(c)
                for i in (set(range(len(pf_trackers))) - matched):
                    pf_trackers[i].invisible_count += 1
                    if is_in_boundary(pf_trackers[i].pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and pf_trackers[i].invisible_count > MAX_INVISIBLE_COUNT:
                        to_del.add(i)
                for c in avail_d:
                    if is_in_boundary(dets[c][:2]+dets[c][2:]/2, W, H, BOUNDARY_PERCENT):
                        pf_trackers.append(Tracker(next_track_id, dets[c], device)); next_track_id += 1
            elif pf_trackers:
                for i, t in enumerate(pf_trackers):
                    t.invisible_count += 1
                    if is_in_boundary(t.pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and t.invisible_count > MAX_INVISIBLE_COUNT: to_del.add(i)
            pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in to_del]

        cv2.line(frame, (int(dp1[0]), int(dp1[1])), (int(dp2[0]), int(dp2[1])), (255, 0, 255), 2)
        for t in pf_trackers:
            if t.invisible_count == 0: t.pf.resample()
            s = t.pf.get_state().cpu().numpy()
            p1, p2 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2)), (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
            clr = (0, 255, 0) if t.invisible_count == 0 else (0, 0, 255)
            cv2.rectangle(frame, p1, p2, clr, 2)
            cv2.putText(frame, f"ID {t.id}", (p1[0], p1[1]-10), 0, 0.7, clr, 2)
        out_video.write(frame)

cap.release()
out_video.release()

In [ ]:
#DETR + Particle filter - ID Switch monitoring
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from transformers import DetrForObjectDetection, DetrImageProcessor
from scipy.optimize import linear_sum_assignment
import supervision as sv
import torchvision
import json
import re

VIDEO_PATH = 'path/to/video.mp4'
MODEL_DIR = 'path/to/detr_model'
COCO_ANNOTATIONS_PATH = 'path/to/annotations.json'
OUTPUT_VIDEO_PATH = 'path/to/output.mp4'

FRAME_SKIP = 1
MAX_SECONDS_TO_PROCESS = 10
DIVIDER_P1 = (0, 104)
DIVIDER_P2 = (496.809, 377.316)
ROAD_ANGLE_NORTH = 28.8
ROAD_ANGLE_SOUTH = 208.8
CONFIDENCE_THRESHOLD = 0.85
NMS_IOU_THRESHOLD = 0.5
MATCHING_ALPHA = 0.4
MATCHING_THRESHOLD = 0.8
MAX_INVISIBLE_COUNT = 6
BOUNDARY_PERCENT = 0.05
NUM_PARTICLES = 150
PROCESS_NOISE_STD = {'x': 1.0, 'y': 1.0, 'vx': 1.0, 'vy': 1.0, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 3
SIGMA_REPULSION = 40.0
ID_SWITCH_IOU_THRESHOLD = 0.3

def extract_frame_number(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[-1]) if nums else -1

def calculate_iou_torch(boxes1, boxes2):
    x1y1_1, x2y2_1 = boxes1[:, :2], boxes1[:, :2] + boxes1[:, 2:]
    x1y1_2, x2y2_2 = boxes2[:, :2], boxes2[:, :2] + boxes2[:, 2:]
    xi1 = torch.max(x1y1_1[:, 0].unsqueeze(1), x1y1_2[:, 0].unsqueeze(0))
    yi1 = torch.max(x1y1_1[:, 1].unsqueeze(1), x1y1_2[:, 1].unsqueeze(0))
    xi2 = torch.min(x2y2_1[:, 0].unsqueeze(1), x2y2_2[:, 0].unsqueeze(0))
    yi2 = torch.min(x2y2_1[:, 1].unsqueeze(1), x2y2_2[:, 1].unsqueeze(0))
    inter_area = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    area1, area2 = boxes1[:, 2] * boxes1[:, 3], boxes2[:, 2] * boxes2[:, 3]
    union_area = area1.unsqueeze(1) + area2.unsqueeze(0) - inter_area
    return torch.nan_to_num(inter_area / (union_area + 1e-6), nan=0.0)

def stabilize_ground_truth_ids(gt_by_frame):
    sorted_frames = sorted(gt_by_frame.keys())
    next_gt_track_id, active_gt_tracks, stabilized_gt = 0, [], {}

    for frame_idx in sorted_frames:
        current_boxes = [x['bbox'] for x in gt_by_frame[frame_idx]]
        stabilized_gt[frame_idx] = []
        if not active_gt_tracks:
            for box in current_boxes:
                stabilized_gt[frame_idx].append({'track_id': next_gt_track_id, 'bbox': box})
                active_gt_tracks.append({'id': next_gt_track_id, 'bbox': box})
                next_gt_track_id += 1
            continue
        curr_t, prev_t = torch.tensor(current_boxes), torch.tensor([t['bbox'] for t in active_gt_tracks])
        if len(curr_t) > 0 and len(prev_t) > 0:
            iou_m = calculate_iou_torch(prev_t, curr_t).numpy()
            r_idx, c_idx = linear_sum_assignment(-iou_m)
            matched_c, new_active = set(), []
            for r, c in zip(r_idx, c_idx):
                if iou_m[r, c] > 0.5:
                    tid, box = active_gt_tracks[r]['id'], current_boxes[c]
                    stabilized_gt[frame_idx].append({'track_id': tid, 'bbox': box})
                    new_active.append({'id': tid, 'bbox': box})
                    matched_c.add(c)
            for c in range(len(current_boxes)):
                if c not in matched_c:
                    box = current_boxes[c]
                    stabilized_gt[frame_idx].append({'track_id': next_gt_track_id, 'bbox': box})
                    new_active.append({'id': next_gt_track_id, 'bbox': box})
                    next_gt_track_id += 1
            active_gt_tracks = new_active
        else:
            active_gt_tracks = []
            for box in current_boxes:
                stabilized_gt[frame_idx].append({'track_id': next_gt_track_id, 'bbox': box})
                active_gt_tracks.append({'id': next_gt_track_id, 'bbox': box})
                next_gt_track_id += 1
    return stabilized_gt

def load_ground_truth_data(json_path):
    with open(json_path, 'r') as f:
        coco = json.load(f)
    imgs = sorted(coco.get('images', []), key=lambda x: extract_frame_number(x['file_name']))
    img_id_map = {img['id']: i for i, img in enumerate(imgs)}
    raw_gt = {}
    for ann in coco.get('annotations', []):
        f_idx = img_id_map.get(ann['image_id'])
        if f_idx is not None:
            if f_idx not in raw_gt: raw_gt[f_idx] = []
            raw_gt[f_idx].append({'bbox': ann['bbox']})
    return stabilize_ground_truth_ids(raw_gt)

class IDSwitchMonitor:
    def __init__(self):
        self.total_id_switches = 0
        self.gt_to_pred_map = {}

    def update(self, frame_idx, trackers, gt_frame):
        if not gt_frame or not trackers: return 0
        p_boxes, p_ids = [], []
        for t in trackers:
            s = t.pf.get_state().cpu().numpy()
            p_boxes.append([s[0]-s[4]/2, s[1]-s[5]/2, s[4], s[5]])
            p_ids.append(t.id)
        g_boxes = [item['bbox'] for item in gt_frame]
        g_ids = [item['track_id'] for item in gt_frame]
        iou_m = calculate_iou_torch(torch.tensor(p_boxes), torch.tensor(g_boxes)).cpu().numpy()
        r_idx, c_idx = linear_sum_assignment(1.0 - iou_m)
        switches = 0
        for r, c in zip(r_idx, c_idx):
            if iou_m[r, c] < ID_SWITCH_IOU_THRESHOLD: continue
            cur_p, cur_g = p_ids[r], g_ids[c]
            if cur_g in self.gt_to_pred_map:
                if self.gt_to_pred_map[cur_g] != cur_p:
                    self.total_id_switches += 1
                    switches += 1
                    self.gt_to_pred_map[cur_g] = cur_p
            else: self.gt_to_pred_map[cur_g] = cur_p
        return switches

def is_in_boundary(pt, W, H, pct):
    bx, by = W * pct, H * pct
    return (pt[0] < bx) or (pt[0] > W - bx) or (pt[1] < by) or (pt[1] > H - by)

def is_north_of_line(pt, p1, p2):
    return (pt[:, 0] - p1[0]) * (p2[1] - p1[1]) - (pt[:, 1] - p1[1]) * (p2[0] - p1[0]) > 0

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_p, proc_std, meas_std, dev):
        self.device, self.num_particles, self.measurement_noise_std = dev, num_p, meas_std
        self.particles = torch.empty((num_p, self.STATE_DIM), device=dev)
        self.weights = torch.ones(num_p, device=dev) / num_p
        self.proc_noise = torch.tensor([proc_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=dev).float()

    def initialize(self, bbox):
        initial = torch.tensor([bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2, 0, 0, bbox[2], bbox[3]], device=self.device).float()
        self.particles = initial + torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self, others, ang_n, ang_s, dp1, dp2):
        is_n = is_north_of_line(self.particles[:, 0:2], dp1, dp2)
        angs = torch.where(is_n, ang_n, ang_s)
        speed = torch.norm(self.particles[:, 2:4], dim=1)
        target_s = torch.where(speed < 0.5, 0.0, 2.0)
        v_des = torch.stack([target_s * torch.cos(angs), target_s * torch.sin(angs)], dim=1)
        v_rep = torch.zeros((self.num_particles, 2), device=self.device)
        if others.numel() > 0:
            vecs = others[:, 0:2].unsqueeze(0) - self.particles[:, 0:2].unsqueeze(1)
            dists = torch.clamp(torch.norm(vecs, dim=2), min=1e-1)
            v_rep = -torch.sum(torch.exp(-(dists**2)/(SIGMA_REPULSION**2)).unsqueeze(2) * (vecs/dists.unsqueeze(2)), dim=1)
        target_v = v_des + v_rep
        self.particles[:, 2:4] = (1 - 0.3) * self.particles[:, 2:4] + 0.3 * torch.nan_to_num(target_v)
        self.particles[:, 0:2] += self.particles[:, 2:4]
        self.particles += torch.randn_like(self.particles) * self.proc_noise

    def update(self, bbox):
        det_c = torch.tensor([bbox[0]+bbox[2]/2, bbox[1]+bbox[3]/2], device=self.device)
        self.weights *= torch.nan_to_num(torch.exp(-0.5 * (torch.norm(self.particles[:, 0:2] - det_c, dim=1)/self.measurement_noise_std)**2), nan=1e-10)
        self.weights = self.weights / (torch.sum(self.weights) + 1e-300)

    def resample(self):
        self.weights = torch.nan_to_num(torch.clamp(self.weights, min=0.0), nan=1.0/self.num_particles)
        s = torch.sum(self.weights)
        w = self.weights/s if s > 0 else torch.ones_like(self.weights)/self.num_particles
        idx = torch.multinomial(w, self.num_particles, replacement=True)
        self.particles, self.weights = self.particles[idx], torch.ones(self.num_particles, device=self.device)/self.num_particles

    def get_state(self):
        w = torch.nan_to_num(self.weights, nan=0.0)
        p = torch.nan_to_num(self.particles, nan=0.0)
        s = torch.sum(w)
        return torch.sum(p * (w/s).unsqueeze(1), dim=0) if s > 0 else torch.mean(p, dim=0)

class Tracker:
    def __init__(self, tid, bbox, dev):
        self.id, self.invisible_count = tid, 0
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, dev)
        self.pf.initialize(bbox)

def run_detector(frame, model, proc, dev):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        inputs = proc(images=rgb, return_tensors='pt').to(dev)
        outputs = model(**inputs)
        res = proc.post_process_object_detection(outputs, threshold=CONFIDENCE_THRESHOLD, target_sizes=torch.tensor([frame.shape[:2]], device=dev))[0]
    if not res['scores'].numel(): return torch.empty((0, 4), device=dev)
    boxes, scores = res['boxes'], res['scores']
    keep = torchvision.ops.nms(boxes, scores, NMS_IOU_THRESHOLD)
    b = boxes[keep]
    return torch.stack([b[:, 0], b[:, 1], b[:, 2]-b[:, 0], b[:, 3]-b[:, 1]], dim=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gt_data = load_ground_truth_data(COCO_ANNOTATIONS_PATH)
id_switch_monitor = IDSwitchMonitor()
model = DetrForObjectDetection.from_pretrained(MODEL_DIR, ignore_mismatched_sizes=True).to(device)
processor = DetrImageProcessor.from_pretrained(MODEL_DIR)
model.eval()

cap = cv2.VideoCapture(VIDEO_PATH)
W, H, source_fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
total_f = min(int(cap.get(7)), int(MAX_SECONDS_TO_PROCESS * source_fps))
out_video = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), source_fps/FRAME_SKIP, (W, H))

ang_n, ang_s = torch.deg2rad(torch.tensor(ROAD_ANGLE_NORTH, device=device)), torch.deg2rad(torch.tensor(ROAD_ANGLE_SOUTH, device=device))
dp1, dp2 = torch.tensor(DIVIDER_P1, device=device), torch.tensor(DIVIDER_P2, device=device)
pf_trackers, next_tid = [], 0

for fidx in tqdm(range(total_f)):
    ret, frame = cap.read()
    if not ret or fidx % FRAME_SKIP != 0: continue

    if fidx == 0:
        dets = torch.tensor([x['bbox'] for x in gt_data.get(fidx, [])], device=device)
        for i in range(dets.shape[0]):
            pf_trackers.append(Tracker(next_tid, dets[i], device)); next_tid += 1
    else:
        dets = run_detector(frame, model, processor, device)
        if pf_trackers:
            states = torch.stack([t.pf.get_state() for t in pf_trackers])
            for i, t in enumerate(pf_trackers):
                others = states[[j for j in range(len(pf_trackers)) if i != j]]
                t.pf.predict(others, ang_n, ang_s, dp1, dp2)

        to_del = set()
        if pf_trackers and dets.numel() > 0:
            st = torch.stack([t.pf.get_state() for t in pf_trackers])
            t_b = torch.stack([st[:,0]-st[:,4]/2, st[:,1]-st[:,5]/2, st[:,4], st[:,5]], dim=1)
            cost = (MATCHING_ALPHA * (1.0 - calculate_iou_torch(t_b, dets))) + \
                   ((1-MATCHING_ALPHA) * (torch.norm(st[:,:2].unsqueeze(1)-(dets[:,:2]+dets[:,2:]/2).unsqueeze(0), dim=2)/torch.tensor([W,H], device=device).norm()))
            r_idx, c_idx = linear_sum_assignment(cost.cpu().numpy())
            matched_t, avail_d = set(), set(range(dets.shape[0]))
            for r, c in zip(r_idx, c_idx):
                if cost[r, c] < MATCHING_THRESHOLD:
                    pf_trackers[r].pf.update(dets[c]); pf_trackers[r].invisible_count = 0
                    matched_t.add(r); avail_d.remove(c)
            for i in (set(range(len(pf_trackers))) - matched_t):
                pf_trackers[i].invisible_count += 1
                if is_in_boundary(pf_trackers[i].pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and pf_trackers[i].invisible_count > MAX_INVISIBLE_COUNT:
                    to_del.add(i)
            for c in avail_d:
                if is_in_boundary(dets[c][:2]+dets[c][2:]/2, W, H, BOUNDARY_PERCENT):
                    pf_trackers.append(Tracker(next_tid, dets[c], device)); next_tid += 1
        pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in to_del]

    id_switch_monitor.update(fidx, pf_trackers, gt_data.get(fidx, []))
    cv2.putText(frame, f"ID Switches: {id_switch_monitor.total_id_switches}", (20, 40), 0, 1, (0,0,255), 2)
    for t in pf_trackers:
        if t.invisible_count == 0: t.pf.resample()
        s = t.pf.get_state().cpu().numpy()
        p1, p2 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2)), (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
        cv2.rectangle(frame, p1, p2, (0,255,0) if t.invisible_count==0 else (0,0,255), 2)
    out_video.write(frame)

cap.release()
out_video.release()

In [ ]:
#Full frame without boundary repulsion - DETR + PARTICLE FILTER
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
import json
import glob
import re

# CONFIGURATION

images_folder = 'insert_path'
json_path = 'insert_path'
output_video_path = 'insert_path'

MATCHING_ALPHA = 0.4
MATCHING_THRESHOLD = 0.8
MAX_INVISIBLE_COUNT = 5
BOUNDARY_PERCENT = 0.02

NUM_PARTICLES = 300
PROCESS_NOISE_STD = {'x': 1.5, 'y': 1.5, 'vx': 1.0, 'vy': 1.0, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 3

# UTILITIES

def extract_frame_number(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[0]) if nums else -1

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def is_in_boundary(center_point_tensor, W, H, percent):
    bound_x, bound_y = W * percent, H * percent
    cx, cy = center_point_tensor[0], center_point_tensor[1]
    return (cx < bound_x) or (cx > W - bound_x) or (cy < bound_y) or (cy > H - bound_y)

# TRACKING ENGINE

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device = device
        self.num_particles = num_particles
        self.measurement_noise_std = measurement_noise_std
        self.particles = torch.empty((self.num_particles, self.STATE_DIM), device=self.device)
        self.weights = torch.ones(self.num_particles, device=self.device) / self.num_particles
        self.proc_noise = torch.tensor([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=self.device).float()

    def initialize(self, bbox_tensor):
        cx = bbox_tensor[0] + bbox_tensor[2]/2
        cy = bbox_tensor[1] + bbox_tensor[3]/2
        w, h = bbox_tensor[2], bbox_tensor[3]
        initial_state = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float()
        self.particles = initial_state + torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self):
        self.particles[:, 0] += self.particles[:, 2]
        self.particles[:, 1] += self.particles[:, 3]
        # Diffusion (Noise)
        noise = torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise
        self.particles += noise

    def update(self, bbox_tensor):
        det_c = torch.tensor([bbox_tensor[0] + bbox_tensor[2]/2, bbox_tensor[1] + bbox_tensor[3]/2], device=self.device)
        distances = torch.norm(self.particles[:, 0:2] - det_c, dim=1)

        # Likelihood with safety epsilon
        likelihood = torch.exp(-0.5 * (distances / self.measurement_noise_std)**2)

        new_weights = self.weights * (likelihood + 1e-10)

        sum_weights = torch.sum(new_weights)
        if sum_weights > 0 and not torch.isnan(sum_weights):
            self.weights = new_weights / sum_weights
        else:
            # Re-initialize weights to uniform if they collapse
            self.weights = torch.ones(self.num_particles, device=self.device) / self.num_particles

    def resample(self):
        # Final check before passing to GPU multinomial kernel
        self.weights = torch.nan_to_num(self.weights, nan=1.0/self.num_particles)
        if torch.sum(self.weights) <= 0:
            self.weights.fill_(1.0 / self.num_particles)

        # Ensure weights sum to exactly 1.0 (requirement for multinomial)
        self.weights = self.weights / torch.sum(self.weights)

        try:
            indices = torch.multinomial(self.weights, self.num_particles, replacement=True)
            self.particles = self.particles[indices]
            self.weights.fill_(1.0 / self.num_particles)
        except RuntimeError:
            self.weights.fill_(1.0 / self.num_particles)

    def get_state(self):
        state = torch.sum(self.particles * self.weights.unsqueeze(1), dim=0)
        return torch.nan_to_num(state)

class Tracker:
    def __init__(self, track_id, initial_bbox_tensor, device):
        self.id = track_id
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, device)
        self.pf.initialize(initial_bbox_tensor)
        self.invisible_count = 0

def calculate_iou_torch(boxes1, boxes2):
    x1y1_1, x2y2_1 = boxes1[:, :2], boxes1[:, :2] + boxes1[:, 2:]
    x1y1_2, x2y2_2 = boxes2[:, :2], boxes2[:, :2] + boxes2[:, 2:]
    xi1 = torch.max(x1y1_1[:, 0].unsqueeze(1), x1y1_2[:, 0].unsqueeze(0))
    yi1 = torch.max(x1y1_1[:, 1].unsqueeze(1), x1y1_2[:, 1].unsqueeze(0))
    xi2 = torch.min(x2y2_1[:, 0].unsqueeze(1), x2y2_2[:, 0].unsqueeze(0))
    yi2 = torch.min(x2y2_1[:, 1].unsqueeze(1), x2y2_2[:, 1].unsqueeze(0))
    inter_area = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    area1, area2 = boxes1[:, 2] * boxes1[:, 3], boxes2[:, 2] * boxes2[:, 3]
    union_area = area1.unsqueeze(1) + area2.unsqueeze(0) - inter_area
    return inter_area / (union_area + 1e-6)

# MAIN EXECUTION
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(json_path, 'r') as f:
    coco_data = json.load(f)

image_id_to_frame_num = {img['id']: extract_frame_number(img['file_name']) for img in coco_data['images']}
detections_by_frame_num = {}

for ann in coco_data['annotations']:
    frame_num = image_id_to_frame_num.get(ann['image_id'], -1)
    if frame_num != -1:
        if frame_num not in detections_by_frame_num:
            detections_by_frame_num[frame_num] = []
        detections_by_frame_num[frame_num].append(ann['bbox'])

image_paths = sorted(glob.glob(os.path.join(images_folder, "*")), key=natural_sort_key)

# Setup VideoWriter with safe check
first_img = cv2.imread(image_paths[0])
if first_img is None:
    raise FileNotFoundError(f"Could not load first image at {image_paths[0]}")
H, W, _ = first_img.shape
out_video = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), 30, (W, H))

pf_trackers = []
next_track_id = 0

for img_path in tqdm(image_paths, desc="Processing Frames"):
    frame = cv2.imread(img_path)
    if frame is None: continue

    current_frame_num = extract_frame_number(os.path.basename(img_path))
    current_dets = detections_by_frame_num.get(current_frame_num, [])
    detections_tensor = torch.tensor(current_dets, dtype=torch.float32).to(device)

    # 1. Predict
    for t in pf_trackers:
        t.pf.predict()

    # 2. Association
    tracks_to_delete = set()
    if pf_trackers and detections_tensor.numel() > 0:
        track_states = torch.stack([t.pf.get_state() for t in pf_trackers])
        track_bboxes = torch.stack([track_states[:,0]-track_states[:,4]/2,
                                    track_states[:,1]-track_states[:,5]/2,
                                    track_states[:,4], track_states[:,5]], axis=1)

        # Cost Calculations
        iou_matrix = 1.0 - calculate_iou_torch(track_bboxes, detections_tensor)

        # Distance Logic
        det_centers = detections_tensor[:, :2] + detections_tensor[:, 2:] / 2.0
        dist_matrix = torch.norm(track_states[:, :2].unsqueeze(1) - det_centers.unsqueeze(0), dim=2)

        diag_norm = torch.tensor([W, H], device=device, dtype=torch.float32).norm()
        dist_cost = dist_matrix / diag_norm

        combined_cost = (MATCHING_ALPHA * iou_matrix) + ((1 - MATCHING_ALPHA) * dist_cost)
        row_ind, col_ind = linear_sum_assignment(combined_cost.cpu().numpy())

        matched_tracks, matched_dets = set(), set()
        for r, c in zip(row_ind, col_ind):
            if combined_cost[r, c] < MATCHING_THRESHOLD:
                pf_trackers[r].pf.update(detections_tensor[c])
                pf_trackers[r].invisible_count = 0
                matched_tracks.add(r)
                matched_dets.add(c)

        # Update visibility and boundary checks
        for i in range(len(pf_trackers)):
            if i not in matched_tracks:
                pf_trackers[i].invisible_count += 1
                if is_in_boundary(pf_trackers[i].pf.get_state()[:2], W, H, BOUNDARY_PERCENT):
                    if pf_trackers[i].invisible_count > MAX_INVISIBLE_COUNT:
                        tracks_to_delete.add(i)

        for i in range(detections_tensor.shape[0]):
            if i not in matched_dets:
                center = detections_tensor[i][:2] + detections_tensor[i][2:]/2
                if is_in_boundary(center, W, H, BOUNDARY_PERCENT):
                    pf_trackers.append(Tracker(next_track_id, detections_tensor[i], device))
                    next_track_id += 1

    elif detections_tensor.numel() > 0:
        for i in range(detections_tensor.shape[0]):
            pf_trackers.append(Tracker(next_track_id, detections_tensor[i], device))
            next_track_id += 1

    # 3. Finalize Frame
    pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in tracks_to_delete]
    for t in pf_trackers:
        if t.invisible_count == 0: t.pf.resample()
        s = t.pf.get_state().cpu().numpy()
        x1, y1 = int(s[0]-s[4]/2), int(s[1]-s[5]/2)
        x2, y2 = int(s[0]+s[4]/2), int(s[1]+s[5]/2)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {t.id}", (x1, y1-5), 0, 0.7, (0, 255, 0), 2)

    out_video.write(frame)

out_video.release()
print(f"Success! Results saved to {output_video_path}")

In [ ]:
#Full Frame Boundary Repulsion
import os
import cv2
import torch
import numpy as np
import json
import glob
import re
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment

# CONFIGURATION
IMAGES_FOLDER = '<Images_folder>'
JSON_PATH = '<Annotations_path>'
OUTPUT_VIDEO_PATH = '<Output_video_path>'
ROAD_BOUNDARY_ANNOTATION_PATH = '<Boundary_annotations_path>'

MATCHING_ALPHA = 0.4
MATCHING_THRESHOLD = 0.8
MAX_INVISIBLE_COUNT = 5
BOUNDARY_PERCENT = 0.05

NUM_PARTICLES = 300
PROCESS_NOISE_STD = {'x': 1, 'y': 1, 'vx': 0.8, 'vy': 0.8, 'w': 0.5, 'h': 0.5}
MEASUREMENT_NOISE_STD = 2

BOUNDARY_REPULSION_K = 100.0
BOUNDARY_REPULSION_DMAX = 45.0

# UTILITIES

def extract_frame_number(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[0]) if nums else -1

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def is_in_boundary(center_point_tensor, W, H, percent):
    bound_x, bound_y = W * percent, H * percent
    cx, cy = center_point_tensor[0], center_point_tensor[1]
    return (cx < bound_x) or (cx > W - bound_x) or (cy < bound_y) or (cy > H - bound_y)

def load_road_boundaries(annotation_path, device):
    with open(annotation_path, 'r') as f:
        coco = json.load(f)

    boundaries = []
    for ann in coco.get("annotations", []):
        for seg in ann.get("segmentation", []):
            pts = np.array(seg).reshape(-1, 2)
            # Forward path only
            pts = pts[:(len(pts) // 2) + 1]
            boundaries.append(pts)

    # Convert polylines to individual line segments
    segments = []
    for boundary in boundaries:
        for i in range(len(boundary) - 1):
            segments.append([boundary[i], boundary[i + 1]])

    if not segments:
        raise ValueError(f"No boundary segments found in {annotation_path}.")

    segments_tensor = torch.tensor(np.array(segments, dtype=np.float32), device=device)
    print(f"Loaded {len(segments)} road boundary segments.")
    return segments_tensor

# CORE CLASSES & FUNCTIONS

class ParticleFilter:
    STATE_DIM = 6

    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device = device
        self.num_particles = num_particles
        self.measurement_noise_std = measurement_noise_std

        self.particles = torch.empty((self.num_particles, self.STATE_DIM), device=self.device)
        self.weights = torch.ones(self.num_particles, device=self.device) / self.num_particles
        self.proc_noise = torch.tensor(
            [process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']],
            device=self.device
        ).float()

    def initialize(self, bbox_tensor):
        cx = bbox_tensor[0] + bbox_tensor[2] / 2
        cy = bbox_tensor[1] + bbox_tensor[3] / 2
        w, h = bbox_tensor[2], bbox_tensor[3]

        initial_state = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float()
        self.particles = initial_state + torch.randn(
            (self.num_particles, self.STATE_DIM), device=self.device
        ) * self.proc_noise

    def predict(self):
        # Apply velocity and process noise
        self.particles[:, 0] += self.particles[:, 2]
        self.particles[:, 1] += self.particles[:, 3]
        noise = torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise
        self.particles += noise

    def apply_boundary_repulsion(self, boundary_segments, k, d_max):
        """Applies a perpendicular repulsive force to particles near road boundaries."""
        pos = self.particles[:, 0:2]

        a = boundary_segments[:, 0, :]
        b = boundary_segments[:, 1, :]
        ab = b - a

        pos_exp = pos.unsqueeze(1)
        a_exp = a.unsqueeze(0)
        ab_exp = ab.unsqueeze(0)

        # Calculate closest point on each segment to each particle
        ap = pos_exp - a_exp
        t = (ap * ab_exp).sum(dim=2) / ((ab_exp * ab_exp).sum(dim=2) + 1e-8)
        t = torch.clamp(t, 0.0, 1.0)
        closest = a_exp + t.unsqueeze(2) * ab_exp

        # Calculate force vectors within influence radius
        diff = pos_exp - closest
        dist = torch.norm(diff, dim=2).clamp(min=1e-8)
        in_range = dist < d_max

        force_mag = torch.zeros_like(dist)
        force_mag[in_range] = k * (1.0 / dist[in_range] - 1.0 / d_max) ** 2

        unit_diff = diff / dist.unsqueeze(2)
        force_vecs = force_mag.unsqueeze(2) * unit_diff
        total_force = force_vecs.sum(dim=1)

        # Apply displacement and velocity adjustments
        self.particles[:, 0:2] += total_force
        self.particles[:, 2:4] += total_force * 0.5

    def update(self, bbox_tensor):
        det_c = torch.tensor(
            [bbox_tensor[0] + bbox_tensor[2] / 2, bbox_tensor[1] + bbox_tensor[3] / 2],
            device=self.device
        )

        # Calculate likelihood based on distance to detection
        distances = torch.norm(self.particles[:, 0:2] - det_c, dim=1)
        likelihood = torch.exp(-0.5 * (distances / self.measurement_noise_std) ** 2)

        new_weights = self.weights * (likelihood + 1e-10)
        sum_weights = torch.sum(new_weights)

        if sum_weights > 0 and not torch.isnan(sum_weights):
            self.weights = new_weights / sum_weights
        else:
            self.weights = torch.ones(self.num_particles, device=self.device) / self.num_particles

    def resample(self):
        self.weights = torch.nan_to_num(self.weights, nan=1.0 / self.num_particles)
        if torch.sum(self.weights) <= 0:
            self.weights.fill_(1.0 / self.num_particles)

        self.weights = self.weights / torch.sum(self.weights)

        try:
            indices = torch.multinomial(self.weights, self.num_particles, replacement=True)
            self.particles = self.particles[indices]
            self.weights.fill_(1.0 / self.num_particles)
        except RuntimeError:
            self.weights.fill_(1.0 / self.num_particles)

    def get_state(self):
        state = torch.sum(self.particles * self.weights.unsqueeze(1), dim=0)
        return torch.nan_to_num(state)

class Tracker:
    def __init__(self, track_id, initial_bbox_tensor, device):
        self.id = track_id
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, device)
        self.pf.initialize(initial_bbox_tensor)
        self.invisible_count = 0

def calculate_iou_torch(boxes1, boxes2):
    x1y1_1, x2y2_1 = boxes1[:, :2], boxes1[:, :2] + boxes1[:, 2:]
    x1y1_2, x2y2_2 = boxes2[:, :2], boxes2[:, :2] + boxes2[:, 2:]

    xi1 = torch.max(x1y1_1[:, 0].unsqueeze(1), x1y1_2[:, 0].unsqueeze(0))
    yi1 = torch.max(x1y1_1[:, 1].unsqueeze(1), x1y1_2[:, 1].unsqueeze(0))
    xi2 = torch.min(x2y2_1[:, 0].unsqueeze(1), x2y2_2[:, 0].unsqueeze(0))
    yi2 = torch.min(x2y2_1[:, 1].unsqueeze(1), x2y2_2[:, 1].unsqueeze(0))

    inter_area = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    area1 = boxes1[:, 2] * boxes1[:, 3]
    area2 = boxes2[:, 2] * boxes2[:, 3]

    union_area = area1.unsqueeze(1) + area2.unsqueeze(0) - inter_area
    return inter_area / (union_area + 1e-6)

# MAIN EXECUTION

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading road boundary annotations...")
boundary_segments = load_road_boundaries(ROAD_BOUNDARY_ANNOTATION_PATH, device)

print("Loading and normalizing detection data...")
with open(JSON_PATH, 'r') as f:
    coco_data = json.load(f)

image_id_to_frame_num = {
    img['id']: extract_frame_number(img['file_name'])
    for img in coco_data['images']
}

detections_by_frame_num = {}
for ann in coco_data['annotations']:
    frame_num = image_id_to_frame_num.get(ann['image_id'], -1)
    if frame_num != -1:
        detections_by_frame_num.setdefault(frame_num, []).append(ann['bbox'])

image_paths = sorted(glob.glob(os.path.join(IMAGES_FOLDER, "*")), key=natural_sort_key)

first_img = cv2.imread(image_paths[0])
if first_img is None:
    raise FileNotFoundError(f"Could not load first image at {image_paths[0]}")

H, W, _ = first_img.shape
out_video = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), 30, (W, H))

pf_trackers = []
next_track_id = 0

for img_path in tqdm(image_paths, desc="Processing Frames"):
    frame = cv2.imread(img_path)
    if frame is None:
        continue

    current_frame_num = extract_frame_number(os.path.basename(img_path))
    current_dets = detections_by_frame_num.get(current_frame_num, [])
    detections_tensor = torch.tensor(current_dets, dtype=torch.float32).to(device)

    # 1. Predict and Apply Repulsion
    for t in pf_trackers:
        t.pf.predict()
        t.pf.apply_boundary_repulsion(boundary_segments, BOUNDARY_REPULSION_K, BOUNDARY_REPULSION_DMAX)

    # 2. Track Association
    tracks_to_delete = set()

    if pf_trackers and detections_tensor.numel() > 0:
        track_states = torch.stack([t.pf.get_state() for t in pf_trackers])
        track_bboxes = torch.stack([
            track_states[:, 0] - track_states[:, 4] / 2,
            track_states[:, 1] - track_states[:, 5] / 2,
            track_states[:, 4],
            track_states[:, 5]
        ], axis=1)

        iou_matrix = 1.0 - calculate_iou_torch(track_bboxes, detections_tensor)
        det_centers = detections_tensor[:, :2] + detections_tensor[:, 2:] / 2.0
        dist_matrix = torch.norm(track_states[:, :2].unsqueeze(1) - det_centers.unsqueeze(0), dim=2)

        diag_norm = torch.tensor([W, H], device=device, dtype=torch.float32).norm()
        dist_cost = dist_matrix / diag_norm

        combined_cost = (MATCHING_ALPHA * iou_matrix) + ((1 - MATCHING_ALPHA) * dist_cost)
        row_ind, col_ind = linear_sum_assignment(combined_cost.cpu().numpy())

        matched_tracks, matched_dets = set(), set()
        for r, c in zip(row_ind, col_ind):
            if combined_cost[r, c] < MATCHING_THRESHOLD:
                pf_trackers[r].pf.update(detections_tensor[c])
                pf_trackers[r].invisible_count = 0
                matched_tracks.add(r)
                matched_dets.add(c)

        # Handle unmatched existing trackers
        for i in range(len(pf_trackers)):
            if i not in matched_tracks:
                pf_trackers[i].invisible_count += 1
                if is_in_boundary(pf_trackers[i].pf.get_state()[:2], W, H, BOUNDARY_PERCENT):
                    if pf_trackers[i].invisible_count > MAX_INVISIBLE_COUNT:
                        tracks_to_delete.add(i)

        # Spawn trackers for unmatched detections
        for i in range(detections_tensor.shape[0]):
            if i not in matched_dets:
                center = detections_tensor[i][:2] + detections_tensor[i][2:] / 2
                if is_in_boundary(center, W, H, BOUNDARY_PERCENT):
                    pf_trackers.append(Tracker(next_track_id, detections_tensor[i], device))
                    next_track_id += 1

    elif detections_tensor.numel() > 0:
        for i in range(detections_tensor.shape[0]):
            pf_trackers.append(Tracker(next_track_id, detections_tensor[i], device))
            next_track_id += 1

    # 3. Finalize and Render
    pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in tracks_to_delete]

    for t in pf_trackers:
        if t.invisible_count == 0:
            t.pf.resample()

        s = t.pf.get_state().cpu().numpy()
        x1, y1 = int(s[0] - s[4] / 2), int(s[1] - s[5] / 2)
        x2, y2 = int(s[0] + s[4] / 2), int(s[1] + s[5] / 2)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {t.id}", (x1, y1 - 5), 0, 0.7, (0, 255, 0), 2)

    out_video.write(frame)

out_video.release()
print(f"Success! Results saved to {OUTPUT_VIDEO_PATH}")

In [ ]:
#ID switch counter script
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
import json
import glob
import re

IMAGES_FOLDER = 'path/to/images'
DETECTIONS_JSON_PATH = 'path/to/detections.json'
GROUND_TRUTH_JSON_PATH = 'path/to/ground_truth.json'
OUTPUT_VIDEO_PATH = 'path/to/output.mp4'

MATCHING_ALPHA = 0.4
MATCHING_THRESHOLD = 0.8
MAX_INVISIBLE_COUNT = 5
BOUNDARY_PERCENT = 0.02
NUM_PARTICLES = 300
PROCESS_NOISE_STD = {'x': 1.5, 'y': 1.5, 'vx': 1.0, 'vy': 1.0, 'w': 1, 'h': 1}
MEASUREMENT_NOISE_STD = 3
ID_SWITCH_IOU_THRESHOLD = 0.5

def extract_frame_number(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[0]) if nums else -1

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def is_in_boundary(center_point_tensor, W, H, percent):
    bound_x, bound_y = W * percent, H * percent
    cx, cy = center_point_tensor[0], center_point_tensor[1]
    return (cx < bound_x) or (cx > W - bound_x) or (cy < bound_y) or (cy > H - bound_y)

def calculate_iou_torch(boxes1, boxes2):
    x1y1_1, x2y2_1 = boxes1[:, :2], boxes1[:, :2] + boxes1[:, 2:]
    x1y1_2, x2y2_2 = boxes2[:, :2], boxes2[:, :2] + boxes2[:, 2:]
    xi1 = torch.max(x1y1_1[:, 0].unsqueeze(1), x1y1_2[:, 0].unsqueeze(0))
    yi1 = torch.max(x1y1_1[:, 1].unsqueeze(1), x1y1_2[:, 1].unsqueeze(0))
    xi2 = torch.min(x2y2_1[:, 0].unsqueeze(1), x2y2_2[:, 0].unsqueeze(0))
    yi2 = torch.min(x2y2_1[:, 1].unsqueeze(1), x2y2_2[:, 1].unsqueeze(0))
    inter_area = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)
    area1, area2 = boxes1[:, 2] * boxes1[:, 3], boxes2[:, 2] * boxes2[:, 3]
    union_area = area1.unsqueeze(1) + area2.unsqueeze(0) - inter_area
    return inter_area / (union_area + 1e-6)

def stabilize_ground_truth_ids(gt_by_frame):
    sorted_frames = sorted(gt_by_frame.keys())
    next_gt_track_id = 0
    active_gt_tracks = []
    stabilized_gt = {}

    for frame_idx in sorted_frames:
        current_boxes = gt_by_frame[frame_idx]
        stabilized_gt[frame_idx] = []

        if not active_gt_tracks:
            for box in current_boxes:
                stabilized_gt[frame_idx].append({'track_id': next_gt_track_id, 'bbox': box})
                active_gt_tracks.append({'id': next_gt_track_id, 'bbox': box})
                next_gt_track_id += 1
            continue

        curr_tensor = torch.tensor(current_boxes, dtype=torch.float32)
        prev_tensor = torch.tensor([t['bbox'] for t in active_gt_tracks], dtype=torch.float32)

        if len(curr_tensor) > 0 and len(prev_tensor) > 0:
            iou_matrix = calculate_iou_torch(prev_tensor, curr_tensor).numpy()
            row_ind, col_ind = linear_sum_assignment(-iou_matrix)
            matched_indices, new_active_tracks = set(), []

            for r, c in zip(row_ind, col_ind):
                if iou_matrix[r, c] > 0.5:
                    track_id = active_gt_tracks[r]['id']
                    stabilized_gt[frame_idx].append({'track_id': track_id, 'bbox': current_boxes[c]})
                    new_active_tracks.append({'id': track_id, 'bbox': current_boxes[c]})
                    matched_indices.add(c)

            for c in range(len(current_boxes)):
                if c not in matched_indices:
                    stabilized_gt[frame_idx].append({'track_id': next_gt_track_id, 'bbox': current_boxes[c]})
                    new_active_tracks.append({'id': next_gt_track_id, 'bbox': current_boxes[c]})
                    next_gt_track_id += 1
            active_gt_tracks = new_active_tracks
        else:
            active_gt_tracks = []
            for box in current_boxes:
                stabilized_gt[frame_idx].append({'track_id': next_gt_track_id, 'bbox': box})
                active_gt_tracks.append({'id': next_gt_track_id, 'bbox': box})
                next_gt_track_id += 1
    return stabilized_gt

def load_coco_annotations(json_path, stabilize=False):
    with open(json_path, 'r') as f:
        coco_data = json.load(f)
    image_id_to_frame_num = {img['id']: extract_frame_number(img['file_name']) for img in coco_data['images']}
    data_by_frame = {}
    for ann in coco_data.get('annotations', []):
        frame_num = image_id_to_frame_num.get(ann['image_id'], -1)
        if frame_num != -1:
            if frame_num not in data_by_frame:
                data_by_frame[frame_num] = []
            data_by_frame[frame_num].append(ann['bbox'])
    return stabilize_ground_truth_ids(data_by_frame) if stabilize else data_by_frame

class IDSwitchMonitor:
    def __init__(self):
        self.total_id_switches = 0
        self.gt_to_pred_map = {}

    def update(self, frame_idx, pf_trackers, gt_data_for_frame):
        if not gt_data_for_frame or not pf_trackers: return 0
        pred_boxes, pred_ids = [], []
        for t in pf_trackers:
            s = t.pf.get_state().cpu().numpy()
            pred_boxes.append([s[0]-s[4]/2, s[1]-s[5]/2, s[4], s[5]])
            pred_ids.append(t.id)

        gt_boxes = [item['bbox'] for item in gt_data_for_frame]
        gt_ids = [item['track_id'] for item in gt_data_for_frame]
        iou_matrix = calculate_iou_torch(torch.tensor(pred_boxes, dtype=torch.float32),
                                        torch.tensor(gt_boxes, dtype=torch.float32)).cpu().numpy()
        row_ind, col_ind = linear_sum_assignment(1.0 - iou_matrix)

        frame_switches = 0
        for r, c in zip(row_ind, col_ind):
            if iou_matrix[r, c] < ID_SWITCH_IOU_THRESHOLD: continue
            curr_pred, curr_gt = pred_ids[r], gt_ids[c]
            if curr_gt in self.gt_to_pred_map:
                if self.gt_to_pred_map[curr_gt] != curr_pred:
                    self.total_id_switches += 1
                    frame_switches += 1
                    self.gt_to_pred_map[curr_gt] = curr_pred
            else:
                self.gt_to_pred_map[curr_gt] = curr_pred
        return frame_switches

class ParticleFilter:
    STATE_DIM = 6
    def __init__(self, num_particles, process_noise_std, measurement_noise_std, device):
        self.device = device
        self.num_particles = num_particles
        self.measurement_noise_std = measurement_noise_std
        self.particles = torch.empty((num_particles, self.STATE_DIM), device=device)
        self.weights = torch.ones(num_particles, device=device) / num_particles
        self.proc_noise = torch.tensor([process_noise_std[k] for k in ['x', 'y', 'vx', 'vy', 'w', 'h']], device=device).float()

    def initialize(self, bbox):
        cx, cy, w, h = bbox[0] + bbox[2]/2, bbox[1] + bbox[3]/2, bbox[2], bbox[3]
        self.particles = torch.tensor([cx, cy, 0, 0, w, h], device=self.device).float() + \
                         torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def predict(self):
        self.particles[:, 0] += self.particles[:, 2]
        self.particles[:, 1] += self.particles[:, 3]
        self.particles += torch.randn((self.num_particles, self.STATE_DIM), device=self.device) * self.proc_noise

    def update(self, bbox):
        det_c = torch.tensor([bbox[0] + bbox[2]/2, bbox[1] + bbox[3]/2], device=self.device)
        dist = torch.norm(self.particles[:, 0:2] - det_c, dim=1)
        self.weights *= (torch.exp(-0.5 * (dist / self.measurement_noise_std)**2) + 1e-10)
        self.weights /= (torch.sum(self.weights) + 1e-10)

    def resample(self):
        self.weights = torch.nan_to_num(self.weights, nan=1.0/self.num_particles)
        if torch.sum(self.weights) <= 0: self.weights.fill_(1.0 / self.num_particles)
        indices = torch.multinomial(self.weights / torch.sum(self.weights), self.num_particles, replacement=True)
        self.particles, self.weights = self.particles[indices], torch.ones(self.num_particles, device=self.device) / self.num_particles

    def get_state(self):
        return torch.nan_to_num(torch.sum(self.particles * self.weights.unsqueeze(1), dim=0))

class Tracker:
    def __init__(self, track_id, initial_bbox, device):
        self.id = track_id
        self.pf = ParticleFilter(NUM_PARTICLES, PROCESS_NOISE_STD, MEASUREMENT_NOISE_STD, device)
        self.pf.initialize(initial_bbox)
        self.invisible_count = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gt_data = load_coco_annotations(GROUND_TRUTH_JSON_PATH, stabilize=True)
detections_by_frame_num = load_coco_annotations(DETECTIONS_JSON_PATH, stabilize=False)
id_switch_monitor = IDSwitchMonitor()
image_paths = sorted(glob.glob(os.path.join(IMAGES_FOLDER, "*")), key=natural_sort_key)

first_img = cv2.imread(image_paths[0])
H, W, _ = first_img.shape
out_video = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), 30, (W, H))

pf_trackers, next_track_id = [], 0

for img_path in tqdm(image_paths):
    frame = cv2.imread(img_path)
    if frame is None: continue
    f_num = extract_frame_number(os.path.basename(img_path))
    dets = torch.tensor(detections_by_frame_num.get(f_num, []), dtype=torch.float32).to(device)

    for t in pf_trackers: t.pf.predict()

    to_del = set()
    if pf_trackers and dets.numel() > 0:
        t_states = torch.stack([t.pf.get_state() for t in pf_trackers])
        t_boxes = torch.stack([t_states[:,0]-t_states[:,4]/2, t_states[:,1]-t_states[:,5]/2, t_states[:,4], t_states[:,5]], axis=1)
        iou_c = 1.0 - calculate_iou_torch(t_boxes, dets)
        dist_c = torch.norm(t_states[:, :2].unsqueeze(1) - (dets[:, :2] + dets[:, 2:]/2).unsqueeze(0), dim=2) / torch.tensor([W, H], device=device).float().norm()
        cost = (MATCHING_ALPHA * iou_c) + ((1 - MATCHING_ALPHA) * dist_c)
        r_ind, c_ind = linear_sum_assignment(cost.cpu().numpy())

        m_t, m_d = set(), set()
        for r, c in zip(r_ind, c_ind):
            if cost[r, c] < MATCHING_THRESHOLD:
                pf_trackers[r].pf.update(dets[c])
                pf_trackers[r].invisible_count = 0
                m_t.add(r); m_d.add(c)

        for i, t in enumerate(pf_trackers):
            if i not in m_t:
                t.invisible_count += 1
                if is_in_boundary(t.pf.get_state()[:2], W, H, BOUNDARY_PERCENT) and t.invisible_count > MAX_INVISIBLE_COUNT:
                    to_del.add(i)

        for i in range(dets.shape[0]):
            if i not in m_d and is_in_boundary(dets[i][:2] + dets[i][2:]/2, W, H, BOUNDARY_PERCENT):
                pf_trackers.append(Tracker(next_track_id, dets[i], device)); next_track_id += 1
    elif dets.numel() > 0:
        for d in dets: pf_trackers.append(Tracker(next_track_id, d, device)); next_track_id += 1

    pf_trackers = [t for i, t in enumerate(pf_trackers) if i not in to_del]
    id_switch_monitor.update(f_num, pf_trackers, gt_data.get(f_num, []))

    cv2.putText(frame, f"ID Switches: {id_switch_monitor.total_id_switches}", (20, 50), 0, 1, (0,0,255), 2)
    for t in pf_trackers:
        if t.invisible_count == 0: t.pf.resample()
        s = t.pf.get_state().cpu().numpy()
        p1 = (int(s[0]-s[4]/2), int(s[1]-s[5]/2))
        p2 = (int(s[0]+s[4]/2), int(s[1]+s[5]/2))
        cv2.rectangle(frame, p1, p2, (0,255,0), 2)
        cv2.putText(frame, f"ID {t.id}", (p1[0], p1[1]-5), 0, 0.7, (0,255,0), 2)
    out_video.write(frame)

out_video.release()